In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.stats import chi2_contingency
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# ============================================================
# OUTPUT DIRECTORY (guaranteed save location)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 All results will be saved in:", os.path.abspath(OUTPUT_DIR))


# ============================================================
# Helper function → save BOTH PNG and SVG
# ============================================================
def save_plot(name):
    png_path = os.path.join(OUTPUT_DIR, f"{name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{name}.svg")

    plt.tight_layout()
    plt.savefig(png_path, dpi=300)
    plt.savefig(svg_path)
    plt.close()

    print("Saved plot →", png_path)
    print("Saved plot →", svg_path)


# ============================================================
# Step 1: Load Data
# ============================================================
file_path = "final_filtered_gene_tf_data.csv"
df = pd.read_csv(file_path)


# ============================================================
# Step 2: Data Cleaning
# ============================================================
df = df.drop(columns=["...7"], errors="ignore")
df["TF"] = df["TF"].astype(str).str.upper()
df["TF_Family"] = df["TF_Family"].astype(str)


# ============================================================
# Step 3: Exploratory Data Analysis
# ============================================================

# Regulation distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=df["Regulation"])
plt.title("Regulation Class Distribution")
save_plot("regulation_class_distribution")


# FoldChange by TF Family
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="TF_Family", y="log2FoldChange", hue="Regulation")
plt.xticks(rotation=90)
plt.title("FoldChange Distribution by TF Family")
save_plot("foldchange_by_tf_family")


# Regulation count by tissue
plt.figure(figsize=(8, 4))
sns.countplot(x=df["Tissue"], hue=df["Regulation"])
plt.title("Regulation Count by Tissue")
plt.xticks(rotation=45)
save_plot("regulation_by_tissue")


# TF Family Distribution in LMC vs MEF
if "Cell_Type" in df.columns:
    for cell in ["LMC", "MEF"]:
        cell_df = df[df["Cell_Type"] == cell]

        plt.figure(figsize=(10, 6))
        sns.countplot(
            y=cell_df["TF_Family"],
            hue=cell_df["Regulation"],
            order=cell_df["TF_Family"].value_counts().index,
        )
        plt.title(f"TF Family Distribution in {cell}")
        save_plot(f"tf_family_distribution_{cell}")


# ============================================================
# Step 4: Statistical Analysis (Chi-square)
# ============================================================
contingency_table = pd.crosstab(df["TF_Family"], df["Regulation"])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"🔹 Chi-square test result: chi2={chi2:.4f}, p-value={p:.4e}")


# TF Family statistics
tf_family_up = (
    df[df["Regulation"] == "upregulated"]["TF_Family"]
    .value_counts(normalize=True)
    .reset_index()
)
tf_family_up.columns = ["TF_Family", "Upregulation_Probability"]

tf_family_down = (
    df[df["Regulation"] == "downregulated"]["TF_Family"]
    .value_counts(normalize=True)
    .reset_index()
)
tf_family_down.columns = ["TF_Family", "Downregulation_Probability"]

tf_family_stats = pd.merge(tf_family_up, tf_family_down, on="TF_Family", how="outer").fillna(0)


# ============================================================
# Step 5: Machine Learning – Predict Regulation
# ============================================================
tf_family_counts = df["TF_Family"].value_counts()
df["TF_Family_Encoded"] = df["TF_Family"].map(tf_family_counts)

le = LabelEncoder()
df["Tissue_Encoded"] = le.fit_transform(df["Tissue"])

threshold = np.percentile(df["log2FoldChange"], 70)
df["High_Regulation"] = (df["log2FoldChange"] >= threshold).astype(int)

X = df[["log2FoldChange", "TF_Family_Encoded", "Tissue_Encoded"]]
y = df["High_Regulation"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)


# Random Forest
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("🔹 Random Forest Performance:")
print(classification_report(y_test, y_pred_rf))


# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss",
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("🔹 XGBoost Performance:")
print(classification_report(y_test, y_pred_xgb))


# ============================================================
# Step 6: Feature Importance
# ============================================================
importance_df = pd.DataFrame(
    {"Feature": X.columns, "Importance": xgb_model.feature_importances_}
)

plt.figure(figsize=(6, 4))
sns.barplot(x="Importance", y="Feature", data=importance_df)
plt.title("Feature Importance for High Regulation (XGBoost)")
save_plot("xgboost_feature_importance")


# ============================================================
# Step 7: Save Results
# ============================================================
ml_csv = os.path.join(OUTPUT_DIR, "ml_results_with_predictions.csv")
tf_csv = os.path.join(OUTPUT_DIR, "tf_family_enrichment_analysis.csv")

df["Prediction_RF"] = rf_model.predict(X_scaled)
df["Prediction_XGB"] = xgb_model.predict(X_scaled)

df.to_csv(ml_csv, index=False)
tf_family_stats.to_csv(tf_csv, index=False)

print("Saved CSV →", ml_csv)
print("Saved CSV →", tf_csv)

print("\n✅ Analysis complete.")
print("📂 Open this folder to see ALL outputs:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# SAME OUTPUT DIRECTORY AS PREVIOUS SCRIPT
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 Figures will be saved in:", os.path.abspath(OUTPUT_DIR))


# ============================================================
# Helper function to save BOTH PNG and SVG
# ============================================================
def save_plot(base_name):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")

    plt.tight_layout()
    plt.savefig(png_path, dpi=300)
    plt.savefig(svg_path)
    plt.close()

    print("✅ Saved:", png_path)
    print("✅ Saved:", svg_path)


# ============================================================
# Load data
# ============================================================
df = pd.read_csv("final_filtered_gene_tf_data.csv")

# Ensure correct column types
df["TF_Family"] = df["TF_Family"].astype(str)
df["Regulation"] = df["Regulation"].astype(str)
df["Tissue"] = df["Tissue"].astype(str).str.upper()


# ============================================================
# Generate boxplots for each tissue
# ============================================================
for tissue in df["Tissue"].unique():

    tissue_df = df[df["Tissue"] == tissue]

    plt.figure(figsize=(14, 6))
    sns.boxplot(
        data=tissue_df,
        x="TF_Family",
        y="log2FoldChange",
        hue="Regulation"
    )

    plt.title(f"log2FoldChange by TF Family in {tissue}")
    plt.xticks(rotation=90)

    save_plot(f"boxplot_log2fc_by_tf_family_{tissue}")


print("\n🎉 All tissue-specific boxplots saved to:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import binomtest, norm
from statsmodels.stats.multitest import multipletests
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist

# Load Data
file_path = "final_filtered_gene_tf_data.csv"
df = pd.read_csv(file_path)

# Count TF Family Occurrences
tf_family_counts = df.groupby(["Tissue", "Regulation", "TF_Family"]).size().reset_index(name="Count")
pivot = tf_family_counts.pivot_table(index=["Tissue", "TF_Family"], columns="Regulation", values="Count", fill_value=0)

# Calculate log2(odds ratio), CI, and p-values
results = []
for (tissue, tf), row in pivot.iterrows():
    up = int(row["upregulated"])
    down = int(row["downregulated"])
    odds_ratio = (up + 1) / (down + 1)
    log2_or = np.log2(odds_ratio)
    se_log_or = np.sqrt(1 / (up + 1) + 1 / (down + 1))
    z = norm.ppf(0.975)
    ci_lower = log2_or - z * se_log_or
    ci_upper = log2_or + z * se_log_or
    total = up + down
    p = binomtest(up, total, p=0.5).pvalue if total > 0 else 1.0
    results.append({
        "Tissue": tissue,
        "TF_Family": tf,
        "Up": up,
        "Down": down,
        "Total": total,
        "Odds_Up": odds_ratio,
        "log2_OR": log2_or,
        "log2_OR_CI_L": ci_lower,
        "log2_OR_CI_U": ci_upper,
        "p-value": p
    })

# Adjust p-values and create DataFrame
results_df = pd.DataFrame(results)
results_df["p-adj"] = multipletests(results_df["p-value"], method="fdr_bh")[1]

# Save all results
results_df.to_csv("tf_family_binomial_results.csv", index=False)

# Plot 1: Barplot of significant TFs by Odds Ratio per Tissue
significant = results_df[results_df["p-adj"] < 0.05]
unique_tissues = significant["Tissue"].unique()
fig, axes = plt.subplots(1, len(unique_tissues), figsize=(7 * len(unique_tissues), 6))

for i, tissue in enumerate(unique_tissues):
    ax = axes[i] if len(unique_tissues) > 1 else axes
    data = significant[significant["Tissue"] == tissue].sort_values("Odds_Up", ascending=False)
    sns.barplot(data=data, x="TF_Family", y="Odds_Up", ax=ax, palette="Set2")
    ax.set_title(f"{tissue} – Significant TF Families", fontsize=14)
    ax.set_ylabel("Odds Ratio (Up / Down)", fontsize=12)
    ax.set_xlabel("TF Family", fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    for container in ax.containers:
        ax.bar_label(container, labels=[f"{val:.2e}" for val in container.datavalues],
                     label_type='edge', fontsize=8, rotation=90, padding=3)
plt.tight_layout()
plt.savefig("plot1_odds_ratio_bars.png", dpi=300)
plt.show()

# Plot 2: Pointplot with 95% CI of log2(OR)
tf_union = significant["TF_Family"].unique()
plot_df = results_df[results_df["TF_Family"].isin(tf_union)].copy()
avg_order = plot_df.groupby("TF_Family")["log2_OR"].mean().sort_values().index
plot_df["TF_Family"] = pd.Categorical(plot_df["TF_Family"], categories=avg_order, ordered=True)
plot_df = plot_df.sort_values(["TF_Family", "Tissue"])

plt.figure(figsize=(14, 6))
sns.set(style="whitegrid")
ax = sns.pointplot(data=plot_df, x="TF_Family", y="log2_OR", hue="Tissue",
                   dodge=0.4, join=False, palette="Set2", errorbar=None)
for i, row in plot_df.iterrows():
    xpos = list(avg_order).index(row["TF_Family"])
    offset = -0.15 if row["Tissue"] == "LMC" else 0.15
    ax.plot([xpos + offset, xpos + offset], [row["log2_OR_CI_L"], row["log2_OR_CI_U"]],
            color="gray", linewidth=1)
plt.axhline(0, color="black", linestyle="--")
plt.xticks(rotation=45, ha="right")
plt.title("TF Family Log2 Odds of Up vs Down Regulation (with 95% CI)", fontsize=14)
plt.ylabel("log2(Odds Ratio Up/Down)", fontsize=12)
plt.xlabel("TF Family", fontsize=12)
plt.legend(title="Tissue", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("plot2_log2or_with_ci.png", dpi=300)
plt.show()

# Plot 3: Clustered Heatmap
clust_df = results_df[results_df["TF_Family"].isin(tf_union)]
heatmap_data = clust_df.pivot(index="TF_Family", columns="Tissue", values="log2_OR").fillna(0)
sig_mask = clust_df.pivot(index="TF_Family", columns="Tissue", values="p-adj") >= 0.05
heatmap_mask = sig_mask.reindex_like(heatmap_data).fillna(True)

row_linkage = linkage(pdist(heatmap_data.values), method="average")
ordered_indices = leaves_list(row_linkage)
heatmap_data = heatmap_data.iloc[ordered_indices]
heatmap_mask = heatmap_mask.iloc[ordered_indices]

cluster_assignments = fcluster(row_linkage, 3, criterion='maxclust')
clustered_tf_families = pd.DataFrame({
    "TF_Family": heatmap_data.index,
    "Cluster": cluster_assignments
}).set_index("TF_Family")
row_labels = [f"{tf} (C{clustered_tf_families.loc[tf, 'Cluster']})" for tf in heatmap_data.index]
heatmap_data.index = row_labels
heatmap_mask.index = row_labels

plt.figure(figsize=(12, len(heatmap_data) * 0.5 + 2))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.6,
    linecolor='gray',
    mask=heatmap_mask,
    cbar_kws={'label': 'log₂(Odds Up/Down)'}
)
plt.title("Clustered Heatmap of TF Family Regulation Patterns (with Cluster Labels)", fontsize=14)
plt.xlabel("Tissue", fontsize=12)
plt.ylabel("TF Family (Cluster ID)", fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig("plot3_clustered_heatmap.png", dpi=300)

("plot1_odds_ratio_bars.png",
 "plot2_log2or_with_ci.png",
 "plot3_clustered_heatmap.png",
 "tf_family_binomial_results.csv")


In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import binomtest, norm
from statsmodels.stats.multitest import multipletests
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist

# ============================================================
# OUTPUT DIRECTORY (same as your earlier scripts)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))


def save_figure(base_name, dpi=300):
    """
    Save current matplotlib figure as BOTH PNG and SVG in OUTPUT_DIR.
    """
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")

    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()

    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)


# ============================================================
# Load Data
# ============================================================
file_path = "final_filtered_gene_tf_data.csv"
df = pd.read_csv(file_path)

# Count TF Family Occurrences
tf_family_counts = (
    df.groupby(["Tissue", "Regulation", "TF_Family"])
      .size()
      .reset_index(name="Count")
)

pivot = tf_family_counts.pivot_table(
    index=["Tissue", "TF_Family"],
    columns="Regulation",
    values="Count",
    fill_value=0
)

# Calculate log2(odds ratio), CI, and p-values
results = []
z = norm.ppf(0.975)

for (tissue, tf), row in pivot.iterrows():
    up = int(row.get("upregulated", 0))
    down = int(row.get("downregulated", 0))

    odds_ratio = (up + 1) / (down + 1)  # pseudo-count for stability
    log2_or = np.log2(odds_ratio)

    se_log_or = np.sqrt(1 / (up + 1) + 1 / (down + 1))
    ci_lower = log2_or - z * se_log_or
    ci_upper = log2_or + z * se_log_or

    total = up + down
    pval = binomtest(up, total, p=0.5).pvalue if total > 0 else 1.0

    results.append({
        "Tissue": tissue,
        "TF_Family": tf,
        "Up": up,
        "Down": down,
        "Total": total,
        "Odds_Up": odds_ratio,
        "log2_OR": log2_or,
        "log2_OR_CI_L": ci_lower,
        "log2_OR_CI_U": ci_upper,
        "p-value": pval
    })

# Adjust p-values and create DataFrame
results_df = pd.DataFrame(results)
results_df["p-adj"] = multipletests(results_df["p-value"], method="fdr_bh")[1]

# Save all results CSV into same folder
csv_path = os.path.join(OUTPUT_DIR, "tf_family_binomial_results.csv")
results_df.to_csv(csv_path, index=False)
print("✅ Saved CSV:", csv_path)

# ============================================================
# Significant results subset
# ============================================================
significant = results_df[results_df["p-adj"] < 0.05].copy()
unique_tissues = significant["Tissue"].unique()

# If no significant results, avoid crashing; still finish cleanly.
if len(significant) == 0:
    print("⚠️ No TF families passed FDR < 0.05. Skipping plots 1–3.")
    print("✅ Done. (CSV still saved.)")
else:
    # ============================================================
    # Plot 1: Barplot of significant TFs by Odds Ratio per Tissue
    # ============================================================
    fig, axes = plt.subplots(1, len(unique_tissues), figsize=(7 * len(unique_tissues), 6))
    if len(unique_tissues) == 1:
        axes = [axes]

    for i, tissue in enumerate(unique_tissues):
        ax = axes[i]
        data = significant[significant["Tissue"] == tissue].sort_values("Odds_Up", ascending=False)

        sns.barplot(data=data, x="TF_Family", y="Odds_Up", ax=ax)
        ax.set_title(f"{tissue} – Significant TF Families", fontsize=14)
        ax.set_ylabel("Odds Ratio (Up / Down)", fontsize=12)
        ax.set_xlabel("TF Family", fontsize=12)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

        # Label bars with scientific notation
        for container in ax.containers:
            ax.bar_label(
                container,
                labels=[f"{val:.2e}" for val in container.datavalues],
                label_type='edge',
                fontsize=8,
                rotation=90,
                padding=3
            )

    save_figure("plot1_odds_ratio_bars")

    # ============================================================
    # Plot 2: Pointplot with 95% CI of log2(OR)
    # ============================================================
    tf_union = significant["TF_Family"].unique()
    plot_df = results_df[results_df["TF_Family"].isin(tf_union)].copy()

    avg_order = plot_df.groupby("TF_Family")["log2_OR"].mean().sort_values().index
    plot_df["TF_Family"] = pd.Categorical(plot_df["TF_Family"], categories=avg_order, ordered=True)
    plot_df = plot_df.sort_values(["TF_Family", "Tissue"])

    plt.figure(figsize=(14, 6))
    sns.set(style="whitegrid")
    ax = sns.pointplot(
        data=plot_df,
        x="TF_Family",
        y="log2_OR",
        hue="Tissue",
        dodge=0.4,
        join=False,
        errorbar=None
    )

    # Draw CI lines manually. Use stable per-tissue offsets (works for 2+ tissues).
    tissues = list(plot_df["Tissue"].unique())
    offsets = np.linspace(-0.2, 0.2, num=len(tissues))
    tissue_to_offset = {t: offsets[i] for i, t in enumerate(tissues)}

    for _, row in plot_df.iterrows():
        xpos = list(avg_order).index(row["TF_Family"])
        offset = tissue_to_offset.get(row["Tissue"], 0.0)
        ax.plot(
            [xpos + offset, xpos + offset],
            [row["log2_OR_CI_L"], row["log2_OR_CI_U"]],
            color="gray",
            linewidth=1
        )

    plt.axhline(0, color="black", linestyle="--")
    plt.xticks(rotation=45, ha="right")
    plt.title("TF Family Log2 Odds of Up vs Down Regulation (with 95% CI)", fontsize=14)
    plt.ylabel("log2(Odds Ratio Up/Down)", fontsize=12)
    plt.xlabel("TF Family", fontsize=12)
    plt.legend(title="Tissue", bbox_to_anchor=(1.01, 1), loc="upper left")

    save_figure("plot2_log2or_with_ci")

    # ============================================================
    # Plot 3: Clustered Heatmap
    # ============================================================
    clust_df = results_df[results_df["TF_Family"].isin(tf_union)].copy()

    heatmap_data = clust_df.pivot(index="TF_Family", columns="Tissue", values="log2_OR").fillna(0)
    p_adj_mat = clust_df.pivot(index="TF_Family", columns="Tissue", values="p-adj")
    heatmap_mask = (p_adj_mat >= 0.05).reindex_like(heatmap_data).fillna(True)

    # Hierarchical clustering on rows
    row_linkage = linkage(pdist(heatmap_data.values), method="average")
    ordered_indices = leaves_list(row_linkage)

    heatmap_data = heatmap_data.iloc[ordered_indices]
    heatmap_mask = heatmap_mask.iloc[ordered_indices]

    # Cluster labels for row annotation
    cluster_assignments = fcluster(row_linkage, 3, criterion="maxclust")
    clustered_tf_families = pd.DataFrame({
        "TF_Family": heatmap_data.index,
        "Cluster": cluster_assignments
    }).set_index("TF_Family")

    row_labels = [f"{tf} (C{clustered_tf_families.loc[tf, 'Cluster']})" for tf in heatmap_data.index]
    heatmap_data.index = row_labels
    heatmap_mask.index = row_labels

    plt.figure(figsize=(12, len(heatmap_data) * 0.5 + 2))
    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        linewidths=0.6,
        linecolor="gray",
        mask=heatmap_mask,
        cbar_kws={"label": "log2(Odds Up/Down)"}
    )

    plt.title("Clustered Heatmap of TF Family Regulation Patterns (with Cluster Labels)", fontsize=14)
    plt.xlabel("Tissue", fontsize=12)
    plt.ylabel("TF Family (Cluster ID)", fontsize=12)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=9)

    save_figure("plot3_clustered_heatmap")

    # Final summary of outputs
    print("\n📦 Outputs created in:", os.path.abspath(OUTPUT_DIR))
    print("   - tf_family_binomial_results.csv")
    print("   - plot1_odds_ratio_bars.(png|svg)")
    print("   - plot2_log2or_with_ci.(png|svg)")
    print("   - plot3_clustered_heatmap.(png|svg)")


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

# Label Encoding
df_model['Gene_encoded'] = LabelEncoder().fit_transform(df_model['Gene'])
df_model['TF_encoded'] = LabelEncoder().fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = LabelEncoder().fit_transform(df_model['Tissue'])

# Subset for stability
df_sample = df_model.sample(n=20000, random_state=42)
X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
y = df_sample['log2FoldChange']

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize model
rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation (on training set)
r2_scores = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
mse_scores = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

# Final training and predictions
rf_model.fit(X_train, y_train)
y_pred_test = rf_model.predict(X_test)

# Test set evaluation
test_r2 = r2_score(y_test, y_pred_test)
test_mse = mean_squared_error(y_test, y_pred_test)

# Print results
print("📊 5-Fold Cross-Validation Results (Train Set)")
print(f"Average R²: {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
print(f"Average MSE: {mse_scores.mean():.4f} ± {mse_scores.std():.4f}")

print("\n🧪 Test Set Evaluation")
print(f"Test R²: {test_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}")

# Feature importance plot
importances = rf_model.feature_importances_
plt.figure(figsize=(6, 4))
sns.barplot(x=importances, y=['Gene', 'TF', 'Tissue'])
plt.title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.show()


In [ ]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# OUTPUT DIRECTORY (same as before)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))


def save_figure(base_name, dpi=300):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")

    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()

    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)


# ============================================================
# Load dataset
# ============================================================
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[["Gene", "TF", "Tissue", "log2FoldChange", "TF_Family"]].dropna()

# ============================================================
# Label Encoding (IMPORTANT: keep encoders if you want to reuse model)
# ============================================================
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model["Gene_encoded"] = gene_le.fit_transform(df_model["Gene"].astype(str))
df_model["TF_encoded"] = tf_le.fit_transform(df_model["TF"].astype(str))
df_model["Tissue_encoded"] = tissue_le.fit_transform(df_model["Tissue"].astype(str))

# ============================================================
# Subset for stability
# - Make sampling safe if dataset has < 20000 rows
# ============================================================
N = 20000
if len(df_model) > N:
    df_sample = df_model.sample(n=N, random_state=42)
else:
    df_sample = df_model.sample(n=len(df_model), random_state=42)

X = df_sample[["Gene_encoded", "TF_encoded", "Tissue_encoded"]]
y = df_sample["log2FoldChange"]

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize model
rf_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# Cross-validation (on training set)
# ============================================================
r2_scores = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="r2")
mse_scores = -cross_val_score(
    rf_model, X_train, y_train, cv=kf, scoring="neg_mean_squared_error"
)

# Final training and predictions
rf_model.fit(X_train, y_train)
y_pred_test = rf_model.predict(X_test)

# Test set evaluation
test_r2 = r2_score(y_test, y_pred_test)
test_mse = mean_squared_error(y_test, y_pred_test)

# Print results
print("\n📊 5-Fold Cross-Validation Results (Train Set)")
print(f"Average R²: {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
print(f"Average MSE: {mse_scores.mean():.4f} ± {mse_scores.std():.4f}")

print("\n🧪 Test Set Evaluation")
print(f"Test R²: {test_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}")

# ============================================================
# Save metrics to CSV (so you don’t lose them)
# ============================================================
metrics_df = pd.DataFrame([{
    "cv_r2_mean": r2_scores.mean(),
    "cv_r2_sd": r2_scores.std(),
    "cv_mse_mean": mse_scores.mean(),
    "cv_mse_sd": mse_scores.std(),
    "test_r2": test_r2,
    "test_mse": test_mse,
    "n_rows_used": len(df_sample),
    "n_features": X.shape[1],
    "n_estimators": rf_model.n_estimators,
    "cv_folds": kf.get_n_splits(),
}])

metrics_path = os.path.join(OUTPUT_DIR, "rf_regression_metrics.csv")
metrics_df.to_csv(metrics_path, index=False)
print("\n✅ Saved metrics CSV:", metrics_path)

# ============================================================
# Save test predictions (handy for diagnostics)
# ============================================================
pred_df = X_test.copy()
pred_df["y_true_log2FoldChange"] = y_test.values
pred_df["y_pred_log2FoldChange"] = y_pred_test

pred_path = os.path.join(OUTPUT_DIR, "rf_regression_test_predictions.csv")
pred_df.to_csv(pred_path, index=False)
print("✅ Saved predictions CSV:", pred_path)

# ============================================================
# Feature importance plot (save PNG + SVG)
# ============================================================
importances = rf_model.feature_importances_
feat_names = ["Gene", "TF", "Tissue"]

plt.figure(figsize=(6, 4))
sns.barplot(x=importances, y=feat_names)
plt.title("Feature Importance (Random Forest Regressor)")
save_figure("rf_regression_feature_importance")

print("\n🎉 Done. All outputs are in:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# OUTPUT DIRECTORY (same as before)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))


def save_figure(base_name, dpi=300):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")

    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()

    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)


# ============================================================
# Load dataset
# ============================================================
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[["Gene", "TF", "Tissue", "log2FoldChange", "TF_Family"]].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model["Gene_encoded"] = gene_le.fit_transform(df_model["Gene"].astype(str))
df_model["TF_encoded"] = tf_le.fit_transform(df_model["TF"].astype(str))
df_model["Tissue_encoded"] = tissue_le.fit_transform(df_model["Tissue"].astype(str))

# Subset for stability (safe if <20000 rows)
N = 20000
if len(df_model) > N:
    df_sample = df_model.sample(n=N, random_state=42)
else:
    df_sample = df_model.sample(n=len(df_model), random_state=42)

X = df_sample[["Gene_encoded", "TF_encoded", "Tissue_encoded"]]
y = df_sample["log2FoldChange"]

# Split into train and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize model
rf_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation (on training set)
r2_scores = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="r2")
mse_scores = -cross_val_score(
    rf_model, X_train, y_train, cv=kf, scoring="neg_mean_squared_error"
)

# Final training and predictions
rf_model.fit(X_train, y_train)
y_pred_test = rf_model.predict(X_test)

# Test set evaluation
test_r2 = r2_score(y_test, y_pred_test)
test_mse = mean_squared_error(y_test, y_pred_test)

# Print results
print("\n📊 5-Fold Cross-Validation Results (Train Set)")
print(f"Average R²: {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
print(f"Average MSE: {mse_scores.mean():.4f} ± {mse_scores.std():.4f}")

print("\n🧪 Test Set Evaluation")
print(f"Test R²: {test_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}")

# ============================================================
# Plot 1: Feature importance plot (SAVE)
# ============================================================
importances = rf_model.feature_importances_
feat_names = ["Gene", "TF", "Tissue"]

plt.figure(figsize=(6, 4))
sns.barplot(x=importances, y=feat_names)
plt.title("Feature Importance (Random Forest Regressor)")
save_figure("rf_regression_feature_importance_v2")


# ============================================================
# 🧬 Average TF Influence Plot (BUG-FIXED + SAVE)
# ============================================================
# Maps
tf_map = (
    df_sample[["TF", "TF_encoded"]]
    .drop_duplicates()
    .set_index("TF_encoded")["TF"]
    .to_dict()
)
tf_family_map = (
    df_sample[["TF", "TF_Family"]]
    .drop_duplicates()
    .set_index("TF")["TF_Family"]
    .to_dict()
)

# Predict on df_sample features ONLY (do not add columns before predicting)
X_features = df_sample[["Gene_encoded", "TF_encoded", "Tissue_encoded"]].copy()
pred_all = rf_model.predict(X_features)

# Build a plotting table
plot_tbl = pd.DataFrame({
    "TF_encoded": df_sample["TF_encoded"].values,
    "Prediction": pred_all
})
plot_tbl["TF"] = plot_tbl["TF_encoded"].map(tf_map)
plot_tbl["TF_Family"] = plot_tbl["TF"].map(tf_family_map)

# Group by TF and compute mean prediction
tf_avg = (
    plot_tbl.groupby(["TF", "TF_Family"], dropna=False)["Prediction"]
    .mean()
    .reset_index()
    .rename(columns={"Prediction": "Average_Prediction"})
    .sort_values("Average_Prediction", ascending=False)
)

# Plot top 20 TFs
top_n = 20
tf_top = tf_avg.head(top_n).copy()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=tf_top,
    x="Average_Prediction",
    y="TF",
    hue="TF_Family",
    dodge=False
)
plt.title("Top 20 TFs by Average Predicted log2FoldChange (Single Run)")
plt.xlabel("Average Predicted log2FoldChange")
plt.ylabel("Transcription Factor")
plt.legend(title="TF Family", bbox_to_anchor=(1.05, 1), loc="upper left")
save_figure("rf_top20_tfs_by_avg_predicted_log2fc")

# Save the table too (useful!)
tf_avg_path = os.path.join(OUTPUT_DIR, "rf_tf_average_prediction_table.csv")
tf_avg.to_csv(tf_avg_path, index=False)
print("✅ Saved table:", tf_avg_path)

print("\n🎉 Done. All outputs are in:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# OUTPUT DIRECTORY (same as before)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))

def save_figure(base_name, dpi=300):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")
    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()
    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)

# ============================================================
# ASSUMES these already exist from your earlier RF code:
# - df_sample (with columns: TF, TF_Family, TF_encoded)
# - X (DataFrame with columns: Gene_encoded, TF_encoded, Tissue_encoded)
# - rf_model (trained RandomForestRegressor)
# ============================================================
required = ["df_sample", "X", "rf_model"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing required objects in memory: "
        + ", ".join(missing)
        + ". Run the RandomForest training code first."
    )

# Map TF_encoded back to TF names + family
tf_map = (
    df_sample[["TF", "TF_encoded"]]
    .drop_duplicates()
    .set_index("TF_encoded")["TF"]
    .to_dict()
)
tf_family_map = (
    df_sample[["TF", "TF_Family"]]
    .drop_duplicates()
    .set_index("TF")["TF_Family"]
    .to_dict()
)

# Build DataFrame with predictions and corresponding TF
# IMPORTANT: predict ONLY on X (3 numeric feature columns)
pred = rf_model.predict(X)

df_analysis = X.copy()
df_analysis["TF"] = df_analysis["TF_encoded"].map(tf_map)
df_analysis["Prediction"] = pred

# Group by TF and compute prediction variability (STD)
tf_pred_stats = (
    df_analysis.groupby("TF")["Prediction"]
    .std()
    .reset_index()
    .rename(columns={"Prediction": "Prediction_STD"})
)

# Add TF family + sort
tf_pred_stats["TF_Family"] = tf_pred_stats["TF"].map(tf_family_map)
tf_pred_stats = tf_pred_stats.sort_values(by="Prediction_STD", ascending=False)

# Save the table (so you can inspect later)
table_path = os.path.join(OUTPUT_DIR, "rf_tf_prediction_std_influence_table.csv")
tf_pred_stats.to_csv(table_path, index=False)
print("✅ Saved table:", table_path)

# Plot top 20 TFs
top_n = 20
plot_df = tf_pred_stats.head(top_n).copy()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=plot_df,
    x="Prediction_STD",
    y="TF",
    hue="TF_Family",
    dodge=False
)
plt.title("Top 20 TFs by Prediction Influence (Random Forest)")
plt.xlabel("Standard Deviation of Predictions (TF Influence)")
plt.ylabel("Transcription Factor")
plt.legend(title="TF Family", bbox_to_anchor=(1.05, 1), loc="upper left")

save_figure("rf_top20_tfs_by_prediction_std_influence")

print("\n🎉 Done. Outputs are in:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# OUTPUT DIRECTORY (same as before)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))


def save_figure(base_name, dpi=300):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")
    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()
    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)


# ============================================================
# Load dataset
# ============================================================
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[["Gene", "TF", "Tissue", "log2FoldChange", "TF_Family"]].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model["Gene_encoded"] = gene_le.fit_transform(df_model["Gene"].astype(str))
df_model["TF_encoded"] = tf_le.fit_transform(df_model["TF"].astype(str))
df_model["Tissue_encoded"] = tissue_le.fit_transform(df_model["Tissue"].astype(str))

# ============================================================
# Multiple iterations
# ============================================================
num_iterations = 10
target_n = 20000

rows_used = min(target_n, len(df_model))
if rows_used < target_n:
    print(f"⚠️ Dataset has only {len(df_model)} rows after dropna; using n={rows_used} per iteration instead of 20000.")

cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

per_iter_rows = []

for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")

    df_sample = df_model.sample(n=rows_used, random_state=i)

    X = df_sample[["Gene_encoded", "TF_encoded", "Tissue_encoded"]]
    y = df_sample["log2FoldChange"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i
    )

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i, n_jobs=-1)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation on training set
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="r2")
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="neg_mean_squared_error")

    cv_r2_scores.append(float(np.mean(r2_cv)))
    cv_mse_scores.append(float(np.mean(mse_cv)))

    # Final training and test evaluation
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(float(r2_test))
    test_mse_scores.append(float(mse_test))

    per_iter_rows.append({
        "iteration": i + 1,
        "n_rows_used": rows_used,
        "cv_r2_mean": float(np.mean(r2_cv)),
        "cv_r2_sd": float(np.std(r2_cv)),
        "cv_mse_mean": float(np.mean(mse_cv)),
        "cv_mse_sd": float(np.std(mse_cv)),
        "test_r2": float(r2_test),
        "test_mse": float(mse_test),
        "random_state": i
    })

# ============================================================
# Summary statistics
# ============================================================
summary = {
    "num_iterations": num_iterations,
    "rows_used_per_iter": rows_used,
    "cv_r2_mean": float(np.mean(cv_r2_scores)),
    "cv_r2_sd": float(np.std(cv_r2_scores)),
    "cv_mse_mean": float(np.mean(cv_mse_scores)),
    "cv_mse_sd": float(np.std(cv_mse_scores)),
    "test_r2_mean": float(np.mean(test_r2_scores)),
    "test_r2_sd": float(np.std(test_r2_scores)),
    "test_mse_mean": float(np.mean(test_mse_scores)),
    "test_mse_sd": float(np.std(test_mse_scores)),
}

print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {summary['cv_r2_mean']:.4f}, Std = {summary['cv_r2_sd']:.4f}")
print(f"Cross-Validation MSE: Mean = {summary['cv_mse_mean']:.4f}, Std = {summary['cv_mse_sd']:.4f}")
print(f"Test R²: Mean = {summary['test_r2_mean']:.4f}, Std = {summary['test_r2_sd']:.4f}")
print(f"Test MSE: Mean = {summary['test_mse_mean']:.4f}, Std = {summary['test_mse_sd']:.4f}")

# ============================================================
# Save metrics to CSV (iteration-level + summary)
# ============================================================
iter_df = pd.DataFrame(per_iter_rows)
iter_csv = os.path.join(OUTPUT_DIR, "rf_regression_multi_iteration_metrics.csv")
iter_df.to_csv(iter_csv, index=False)
print("\n✅ Saved iteration metrics CSV:", iter_csv)

summary_df = pd.DataFrame([summary])
summary_csv = os.path.join(OUTPUT_DIR, "rf_regression_multi_iteration_summary.csv")
summary_df.to_csv(summary_csv, index=False)
print("✅ Saved summary CSV:", summary_csv)

# ============================================================
# Plot: Boxplots for distribution of R² (save PNG + SVG)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")

sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")

save_figure("rf_regression_r2_distributions_boxplots")

print("\n🎉 Done. All outputs are in:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded'] = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded'] = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# Save example input dataframe
df_model.head(50).to_csv("example_input_dataframe.csv", index=False)

# Trackers
num_iterations = 10
cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

final_predictions_df = None
final_training_df = None

# Loop
for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(r2_test)
    test_mse_scores.append(mse_test)

    # Save outputs from final iteration
    if i == num_iterations - 1:
        # Build TF_Family lookup
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        # Test predictions
        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange'] = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene'] = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF'] = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        # Training set
        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene'] = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF'] = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# Summary
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# Save outputs
if final_predictions_df is not None:
    final_predictions_df.to_csv("random_forest_predictions_with_tf_family.csv", index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv("random_forest_training_set.csv", index=False)
    print("✅ Saved: random_forest_training_set.csv")

# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")
sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded'] = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded'] = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# Save example input dataframe
df_model.head(50).to_csv("example_input_dataframe.csv", index=False)

# Trackers
num_iterations = 10
cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

final_predictions_df = None
final_training_df = None

# Loop
for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)

    # Explicitly mark encodings as categorical to avoid implicit ordinal interpretation
    X_train = X_train.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    X_test = X_test.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(r2_test)
    test_mse_scores.append(mse_test)

    # Save outputs from final iteration
    if i == num_iterations - 1:
        # Build TF_Family lookup
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        # Test predictions
        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange'] = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene'] = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF'] = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        # Training set
        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene'] = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF'] = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# Summary
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# Save outputs
if final_predictions_df is not None:
    final_predictions_df.to_csv("random_forest_predictions_with_tf_family.csv", index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv("random_forest_training_set.csv", index=False)
    print("✅ Saved: random_forest_training_set.csv")

# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")
sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded'] = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded'] = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# Save an example of how the dataframe looks
df_model.head(50).to_csv("example_input_dataframe.csv", index=False)

# Initialize trackers for multiple iterations
num_iterations = 10
cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

# Placeholder for saving predictions from final iteration
final_predictions_df = None

for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation on training set
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Final training and test evaluation
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(r2_test)
    test_mse_scores.append(mse_test)

    # Save predictions from final iteration
    if i == num_iterations - 1:
        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange'] = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene'] = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF'] = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])

        # Merge to get TF_Family
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')

        final_predictions_df = X_test_copy

# Summary statistics
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# Save predictions to CSV
if final_predictions_df is not None:
    final_predictions_df.to_csv("random_forest_predictions_with_tf_family.csv", index=False)
    print("\n✅ Final test set predictions saved to: random_forest_predictions_with_tf_family.csv")

# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")
sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

# ✅ Load Data
file_path = "final_filtered_gene_tf_data.csv"
df = pd.read_csv(file_path)

# ✅ Data Cleaning
df = df.drop(columns=["...7"], errors="ignore")  # Drop unnecessary column
df["TF"] = df["TF"].astype(str).str.upper()
df["TF_Family"] = df["TF_Family"].astype(str)
df["Tissue"] = df["Tissue"].astype(str)
df = df.dropna()  # Handle missing values

# ✅ Compute Mean Fold Change for Each TF Family in LMC & MEF
tf_family_lmc = df[df["Tissue"] == "LMC"].groupby("TF_Family")["log2FoldChange"].mean().reset_index()
tf_family_mef = df[df["Tissue"] == "MEF"].groupby("TF_Family")["log2FoldChange"].mean().reset_index()

# Merge to Compare LMC vs. MEF TF Families
tf_family_comparison = tf_family_lmc.merge(tf_family_mef, on="TF_Family", suffixes=("_LMC", "_MEF"))
tf_family_comparison["FoldChange_Diff"] = tf_family_comparison["log2FoldChange_LMC"] - tf_family_comparison["log2FoldChange_MEF"]

# ✅ Identify TF Families with Significant Fold-Change Differences Between LMC & MEF
tf_family_p_values = []
for tf in df["TF_Family"].unique():
    lmc_values = df[(df["TF_Family"] == tf) & (df["Tissue"] == "LMC")]["log2FoldChange"]
    mef_values = df[(df["TF_Family"] == tf) & (df["Tissue"] == "MEF")]["log2FoldChange"]
    
    if len(lmc_values) > 1 and len(mef_values) > 1:
        t_stat, p_val = ttest_ind(lmc_values, mef_values, equal_var=False)
        tf_family_p_values.append({"TF_Family": tf, "t_stat": t_stat, "p_value": p_val})

# Convert to DataFrame
tf_family_p_values_df = pd.DataFrame(tf_family_p_values)
tf_family_p_values_df["Significant"] = tf_family_p_values_df["p_value"] < 0.05  # Mark significant differences

# Merge significance results into comparison table
tf_family_comparison = tf_family_comparison.merge(tf_family_p_values_df, on="TF_Family", how="left")

# ✅ Sort by most significant differences
tf_family_comparison = tf_family_comparison.sort_values(by="p_value")

# ✅ Save Results
tf_family_comparison.to_csv("tf_family_LMC_vs_MEF_statistical_analysis.csv", index=False)

# ✅ Display Top TF Families with Most Significant Differences
print("\n🔹 Top TF Families with Most Significant Differences Between LMC & MEF:")
print(tf_family_comparison.head(20))


In [ ]:
# ✅ Plotting Section: Enhanced Horizontal Bar Plot with Cleaned P-value Labels
top_n = 15
top_df = tf_family_comparison.head(top_n)

# Normalize p-values for color mapping (smaller p = darker)
norm_p = -np.log10(top_df["p_value"])
colors = sns.color_palette("coolwarm", as_cmap=True)(norm_p / norm_p.max())

plt.figure(figsize=(12, 8))
bars = plt.barh(top_df["TF_Family"], top_df["FoldChange_Diff"], color=colors, edgecolor='black')

# Aesthetics
plt.xlabel("Mean Fold Change Difference (LMC - MEF)", fontsize=12)
plt.title("Top 15 TF Families with Most Significant Expression Differences\nBetween LMC and MEF", fontsize=14)
plt.gca().invert_yaxis()

# Annotate bars with nicely formatted p-values
for bar, p_val in zip(bars, top_df["p_value"]):
    xpos = bar.get_width()
    ypos = bar.get_y() + bar.get_height() / 2

    if p_val < 1e-100:
        p_str = "p < 1e-100"
    elif p_val < 1e-10:
        p_str = f"p < {p_val:.0e}"
    else:
        p_str = f"p = {p_val:.1e}"

    # Adjust horizontal alignment based on bar direction
    align = 'left' if xpos >= 0 else 'right'
    offset = 0.02 if xpos >= 0 else -0.02

    plt.text(xpos + offset, ypos, p_str, va='center', ha=align, fontsize=10)

# Style
plt.xticks(fontsize=10)
plt.yticks(fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ✅ Plotting Section: Slim Horizontal Bar Plot with P-values
top_n = 15
top_df = tf_family_comparison.head(top_n)

plt.figure(figsize=(12, 8))

# Use a consistent color for all bars
bar_color = sns.color_palette("coolwarm", 1)[0]

# Slim bar height
bar_height = 0.3
y_positions = np.arange(len(top_df))

bars = plt.barh(y_positions, top_df["FoldChange_Diff"], color=bar_color, edgecolor='black', height=bar_height)

# Y-axis labels (TF Families)
plt.yticks(y_positions, top_df["TF_Family"], fontsize=11)

# Aesthetics
plt.xlabel("Mean Fold Change Difference (LMC - MEF)", fontsize=12)
plt.title("Top 15 TF Families with Most Significant Expression Differences\nBetween LMC and MEF", fontsize=14)
plt.gca().invert_yaxis()

# Annotate bars with p-values (centered on the bars)
for xpos, ypos, p_val in zip(top_df["FoldChange_Diff"], y_positions, top_df["p_value"]):
    if p_val < 1e-100:
        p_str = "p < 1e-100"
    elif p_val < 1e-10:
        p_str = f"p < {p_val:.0e}"
    else:
        p_str = f"p = {p_val:.1e}"

    # Place text in the center of the bar, with contrasting font color
    plt.text(xpos / 2, ypos, p_str, va='center', ha='center', fontsize=10, color='white', fontweight='bold')

# Style
plt.xticks(fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Prepare your top-n DataFrame
top_n = 15
top_df = tf_family_comparison.head(top_n)

# Create figure & axis
fig, ax = plt.subplots(figsize=(12, 8))

# Bar styling
bar_color = '#add8e6'   # pale blue
bar_height = 0.3
y_positions = np.arange(len(top_df))

# Draw horizontal bars
bars = ax.barh(
    y_positions,
    top_df["FoldChange_Diff"],
    color=bar_color,
    edgecolor='black',
    height=bar_height
)

# Axis labels & title
ax.set_yticks(y_positions)
ax.set_yticklabels(top_df["TF_Family"], fontsize=11)
ax.set_xlabel("Mean Fold Change Difference (LMC - MEF)", fontsize=12)
ax.set_title(
    "Top 15 TF Families with Most Significant Expression Differences\nBetween LMC and MEF",
    fontsize=14
)
ax.invert_yaxis()

# Render the canvas so we can get accurate bounding boxes
fig.canvas.draw()
renderer = fig.canvas.get_renderer()

# Annotate p-values below each bar, left-justified
for bar, p_val in zip(bars, top_df["p_value"]):
    # Get bar bbox in display coords
    bbox = bar.get_window_extent(renderer=renderer)
    # Position 5 pixels below the bar
    text_disp_x = bbox.x0       # left edge of bar
    text_disp_y = bbox.y0 - 5   # 5 px below bar bottom

    # Convert back to data coords
    text_data_x, text_data_y = ax.transData.inverted().transform((text_disp_x, text_disp_y))

    # Format p-value string
    if p_val < 1e-100:
        p_str = "p < 1e-100"
    elif p_val < 1e-10:
        p_str = f"p < {p_val:.0e}"
    else:
        p_str = f"p = {p_val:.1e}"

    # Draw text
    ax.text(
        text_data_x, text_data_y, p_str,
        ha='left',   # left-justify at bar start
        va='top',    # anchor the text top at this y
        fontsize=10,
        color='black'
    )

# Final styling
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ── Build a consistent color mapping for TF families ────────────────────────────
# Get the unique TF families in your prediction-influence plot (or in top_df)
unique_fams = tf_pred_stats['TF_Family'].unique()

# Choose a palette (must match the one used in your RF barplot)
palette = sns.color_palette("tab10", len(unique_fams))
fam_to_color = dict(zip(unique_fams, palette))

# ── Fold-change Difference Plot with Family-matched Annotation Colors ──────────
top_n = 15
top_df = tf_family_comparison.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))

# Bars remain pale blue
bar_color = '#add8e6'
bar_height = 0.3
y_positions = np.arange(len(top_df))

bars = ax.barh(
    y_positions,
    top_df["FoldChange_Diff"],
    color=bar_color,
    edgecolor='black',
    height=bar_height
)

# Y-axis labels
ax.set_yticks(y_positions)
ax.set_yticklabels(top_df["TF_Family"], fontsize=11)

# Title & axis labels
ax.set_xlabel("Mean Fold Change Difference (LMC - MEF)", fontsize=12)
ax.set_title(
    "Top 15 TF Families with Most Significant Expression Differences\nBetween LMC and MEF",
    fontsize=14
)
ax.invert_yaxis()

# Annotate each p-value below its bar, using the TF-family color
for bar, fam, p_val in zip(bars, top_df["TF_Family"], top_df["p_value"]):
    # Determine the display offset
    bar_bottom = bar.get_y()
    label_y = bar_bottom - 0.02  # small downward offset in data coords

    # Format p-value
    if p_val < 1e-100:
        p_str = "p < 1e-100"
    elif p_val < 1e-10:
        p_str = f"p < {p_val:.0e}"
    else:
        p_str = f"p = {p_val:.1e}"

    # Draw the text, colored by TF family
    ax.text(
        0, label_y, p_str,
        ha='left', va='top',
        fontsize=10,
        color=fam_to_color[fam]
    )

# Color the y-tick labels by TF family as well
for label in ax.get_yticklabels():
    fam = label.get_text()
    if fam in fam_to_color:
        label.set_color(fam_to_color[fam])

# Final styling
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ── Build a consistent color mapping for TF families ────────────────────────────
unique_fams = tf_pred_stats['TF_Family'].unique()
palette = sns.color_palette("tab10", len(unique_fams))
fam_to_color = dict(zip(unique_fams, palette))

# ── Fold-change Difference Plot with Family-matched Annotation Colors ──────────
top_n = 15
top_df = tf_family_comparison.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))

# Bars remain pale blue
bar_color = '#add8e6'
bar_height = 0.3
y_positions = np.arange(len(top_df))

bars = ax.barh(
    y_positions,
    top_df["FoldChange_Diff"],
    color=bar_color,
    edgecolor='black',
    height=bar_height
)

# Y-axis labels (bold, colored)
ax.set_yticks(y_positions)
yticks = ax.set_yticklabels(top_df["TF_Family"], fontsize=11)
for label in yticks:
    fam = label.get_text()
    if fam in fam_to_color:
        label.set_color(fam_to_color[fam])
    label.set_fontweight('bold')

# Title & axis labels (bold)
ax.set_xlabel("Mean Fold Change Difference (LMC - MEF)", fontsize=12, fontweight='bold')
ax.set_title(
    "Top 15 TF Families with Most Significant Expression Differences\nBetween LMC and MEF",
    fontsize=14,
    fontweight='bold'
)
ax.invert_yaxis()

# Annotate each p-value below its bar, left-aligned, in family color and bold
for bar, fam, p_val in zip(bars, top_df["TF_Family"], top_df["p_value"]):
    # Position text just below the bar (data coords)
    label_y = bar.get_y() + bar.get_height() + 0.02

    # Format p-value
    if p_val < 1e-100:
        p_str = "p < 1e-100"
    elif p_val < 1e-10:
        p_str = f"p < {p_val:.0e}"
    else:
        p_str = f"p = {p_val:.1e}"

    ax.text(
        0, label_y, p_str,
        ha='left', va='bottom',
        fontsize=10,
        color=fam_to_color[fam],
        fontweight='bold'
    )

# Final styling
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ── Build a consistent color mapping for TF families ────────────────────────────
unique_fams = tf_pred_stats['TF_Family'].unique()
palette = sns.color_palette("tab10", len(unique_fams))
fam_to_color = dict(zip(unique_fams, palette))

# ── Fold-change Difference Plot with Family-matched Annotation Colors ──────────
top_n = 15
top_df = tf_family_comparison.head(top_n)

fig, ax = plt.subplots(figsize=(12, 8))

# Bars remain pale blue
bar_color = '#add8e6'
bar_height = 0.3
y_positions = np.arange(len(top_df))

bars = ax.barh(
    y_positions,
    top_df["FoldChange_Diff"],
    color=bar_color,
    edgecolor='black',
    height=bar_height
)

# Y-axis labels (bold, colored)
ax.set_yticks(y_positions)
yticks = ax.set_yticklabels(top_df["TF_Family"], fontsize=11)
for label in yticks:
    fam = label.get_text()
    if fam in fam_to_color:
        label.set_color(fam_to_color[fam])
    label.set_fontweight('bold')

# Title & axis labels (bold)
ax.set_xlabel("Mean Fold Change Difference (LMC - MEF)", fontsize=12, fontweight='bold')
ax.set_title(
    "Top 15 TF Families with Most Significant Expression Differences\nBetween LMC and MEF",
    fontsize=14,
    fontweight='bold'
)
ax.invert_yaxis()

# Draw the canvas so we can compute accurate pixel positions
fig.canvas.draw()
renderer = fig.canvas.get_renderer()

# Annotate p-values directly below each bar
for bar, fam, p_val in zip(bars, top_df["TF_Family"], top_df["p_value"]):
    # get bar bounding box in display (pixel) coordinates
    bbox = bar.get_window_extent(renderer=renderer)
    # choose 5 pixels below bar bottom
    text_disp_x = bbox.x0
    text_disp_y = bbox.y0 - 5

    # convert display coords back to data coords
    text_data_x, text_data_y = ax.transData.inverted().transform((text_disp_x, text_disp_y))

    # format p-value
    if p_val < 1e-100:
        p_str = "p < 1e-100"
    elif p_val < 1e-10:
        p_str = f"p < {p_val:.0e}"
    else:
        p_str = f"p = {p_val:.1e}"

    # place the text
    ax.text(
        text_data_x, text_data_y, p_str,
        ha='left', va='top',
        fontsize=10, color=fam_to_color[fam], fontweight='bold'
    )

# Final styling
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
# Get unique tissues
unique_tissues = df['Tissue'].unique()

# Plot distribution for each tissue
for tissue in unique_tissues:
    plt.figure(figsize=(10, 6))
    
    # Filter data for the specific tissue
    tissue_df = df[df['Tissue'] == tissue]
    
    # Count occurrences of each TF family in this tissue
    tf_family_counts = tissue_df['TF_Family'].value_counts()
    
    # Plot
    tf_family_counts.plot(kind='bar', color='steelblue', edgecolor='black')
    
    # Labels and title
    plt.xlabel('TF Family', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.title(f'Distribution of Transcription Factor Families in {tissue}', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    
    # Show the plot
    plt.show()

In [ ]:
import requests

# Define your TF name (replace with your target TF)
target_tf = "SP1"  # Example: Searching for SP1 TF motifs

# Fetch JASPAR motifs
jaspar_url = "https://jaspar.elixir.no/api/v1/matrix/"
response = requests.get(jaspar_url)

if response.status_code == 200:
    jaspar_data = response.json()
    motifs = jaspar_data.get("results", [])

    print(f"✅ Searching for JASPAR motifs related to '{target_tf}'...\n")

    for motif in motifs:  
        motif_id = motif.get("matrix_id", "N/A")
        tf_name = motif.get("name", "Unknown TF")
        url = motif.get("url", "N/A")

        if target_tf.lower() in tf_name.lower():
            print(f"🔍 Found: ID: {motif_id} | TF: {tf_name} | Link: {url}")
    
else:
    print(f"❌ Failed to retrieve JASPAR motifs. Status Code: {response.status_code}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded'] = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded'] = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# Save example input dataframe
df_model.head(50).to_csv("example_input_dataframe.csv", index=False)

# Trackers
num_iterations = 10
cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

final_predictions_df = None
final_training_df = None

# Loop
for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)

    # Explicitly mark encodings as categorical to avoid implicit ordinal interpretation
    X_train = X_train.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    X_test = X_test.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(r2_test)
    test_mse_scores.append(mse_test)

    # Save outputs from final iteration
    if i == num_iterations - 1:
        # Build TF_Family lookup
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        # Test predictions
        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange'] = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene'] = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF'] = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        # Training set
        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene'] = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF'] = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# Summary
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# Save outputs
if final_predictions_df is not None:
    final_predictions_df.to_csv("random_forest_predictions_with_tf_family.csv", index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv("random_forest_training_set.csv", index=False)
    print("✅ Saved: random_forest_training_set.csv")

# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")
sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")
plt.tight_layout()
plt.show()


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# OUTPUT DIRECTORY (same as before)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))


def save_figure(base_name, dpi=300):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")
    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()
    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)


# ============================================================
# Load dataset
# ============================================================
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[["Gene", "TF", "Tissue", "log2FoldChange", "TF_Family"]].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model["Gene_encoded"] = gene_le.fit_transform(df_model["Gene"].astype(str))
df_model["TF_encoded"] = tf_le.fit_transform(df_model["TF"].astype(str))
df_model["Tissue_encoded"] = tissue_le.fit_transform(df_model["Tissue"].astype(str))

# ============================================================
# Save example input dataframe (TOP 50) into OUTPUT_DIR
# ============================================================
example_path = os.path.join(OUTPUT_DIR, "example_input_dataframe.csv")
df_model.head(50).to_csv(example_path, index=False)
print("✅ Saved:", example_path)

# ============================================================
# Trackers
# ============================================================
num_iterations = 10
target_n = 20000
rows_used = min(target_n, len(df_model))

if rows_used < target_n:
    print(f"⚠️ Dataset has only {len(df_model)} rows after dropna; using n={rows_used} per iteration instead of 20000.")

cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

final_predictions_df = None
final_training_df = None

# Build TF_Family lookup once (stable)
tf_family_map = df_model[["TF", "TF_Family"]].drop_duplicates()

# ============================================================
# Loop
# ============================================================
for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")

    df_sample = df_model.sample(n=rows_used, random_state=i)

    X = df_sample[["Gene_encoded", "TF_encoded", "Tissue_encoded"]]
    y = df_sample["log2FoldChange"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i
    )

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i, n_jobs=-1)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="r2")
    mse_cv = -cross_val_score(
        rf_model, X_train, y_train, cv=kf, scoring="neg_mean_squared_error"
    )

    cv_r2_scores.append(float(np.mean(r2_cv)))
    cv_mse_scores.append(float(np.mean(mse_cv)))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(float(r2_test))
    test_mse_scores.append(float(mse_test))

    # Save outputs from final iteration
    if i == num_iterations - 1:
        # Test predictions with decoded labels
        X_test_copy = X_test.copy()
        X_test_copy["Actual_log2FoldChange"] = y_test.values
        X_test_copy["Predicted_log2FoldChange"] = y_pred_test

        X_test_copy["Gene"] = gene_le.inverse_transform(X_test_copy["Gene_encoded"].astype(int))
        X_test_copy["TF"] = tf_le.inverse_transform(X_test_copy["TF_encoded"].astype(int))
        X_test_copy["Tissue"] = tissue_le.inverse_transform(X_test_copy["Tissue_encoded"].astype(int))

        X_test_copy = X_test_copy.merge(tf_family_map, on="TF", how="left")
        final_predictions_df = X_test_copy

        # Training set with decoded labels
        X_train_copy = X_train.copy()
        X_train_copy["log2FoldChange"] = y_train.values

        X_train_copy["Gene"] = gene_le.inverse_transform(X_train_copy["Gene_encoded"].astype(int))
        X_train_copy["TF"] = tf_le.inverse_transform(X_train_copy["TF_encoded"].astype(int))
        X_train_copy["Tissue"] = tissue_le.inverse_transform(X_train_copy["Tissue_encoded"].astype(int))

        X_train_copy = X_train_copy.merge(tf_family_map, on="TF", how="left")
        final_training_df = X_train_copy


# ============================================================
# Summary
# ============================================================
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# ============================================================
# Save outputs into OUTPUT_DIR
# ============================================================
if final_predictions_df is not None:
    pred_path = os.path.join(OUTPUT_DIR, "random_forest_predictions_with_tf_family.csv")
    final_predictions_df.to_csv(pred_path, index=False)
    print("✅ Saved:", pred_path)

if final_training_df is not None:
    train_path = os.path.join(OUTPUT_DIR, "random_forest_training_set.csv")
    final_training_df.to_csv(train_path, index=False)
    print("✅ Saved:", train_path)

# Also save iteration metrics (super useful later)
iter_metrics = pd.DataFrame({
    "iteration": np.arange(1, num_iterations + 1),
    "cv_r2": cv_r2_scores,
    "cv_mse": cv_mse_scores,
    "test_r2": test_r2_scores,
    "test_mse": test_mse_scores
})
iter_metrics_path = os.path.join(OUTPUT_DIR, "rf_iteration_metrics.csv")
iter_metrics.to_csv(iter_metrics_path, index=False)
print("✅ Saved:", iter_metrics_path)

# ============================================================
# Boxplots (save PNG + SVG)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")

sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")

save_figure("rf_r2_distributions_with_iterations")

print("\n🎉 Done. All outputs are in:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded'] = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded'] = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# Save example input dataframe
df_model.head(50).to_csv("example_input_dataframe.csv", index=False)

# Trackers
num_iterations = 10
cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []
feature_importance_list = []

final_predictions_df = None
final_training_df = None

# Loop
for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)

    # Explicitly mark encodings as categorical to avoid implicit ordinal interpretation
    X_train = X_train.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    X_test = X_test.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(r2_test)
    test_mse_scores.append(mse_test)

    # Track feature importances
    feature_importance_list.append(rf_model.feature_importances_)

    # Save outputs from final iteration
    if i == num_iterations - 1:
        # Build TF_Family lookup
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        # Test predictions
        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange'] = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene'] = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF'] = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        # Training set
        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene'] = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF'] = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# Summary
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# Save outputs
if final_predictions_df is not None:
    final_predictions_df.to_csv("random_forest_predictions_with_tf_family.csv", index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv("random_forest_training_set.csv", index=False)
    print("✅ Saved: random_forest_training_set.csv")

# Boxplots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")
sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")
plt.tight_layout()
plt.show()

# Plot consensus feature importance
avg_importances = np.mean(np.array(feature_importance_list), axis=0)
feature_names = ['Gene_encoded', 'TF_encoded', 'Tissue_encoded']

plt.figure(figsize=(6, 4))
sns.barplot(x=avg_importances, y=feature_names)
plt.title("Consensus Feature Importance (Across 10 Iterations)")
plt.xlabel("Average Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import mean_absolute_error

# Holdout gene-based split
unique_genes = df_model['Gene_encoded'].unique()
np.random.seed(42)
test_genes = np.random.choice(unique_genes, size=int(0.2 * len(unique_genes)), replace=False)

df_train = df_model[~df_model['Gene_encoded'].isin(test_genes)]
df_test = df_model[df_model['Gene_encoded'].isin(test_genes)]

X_train = df_train[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
y_train = df_train['log2FoldChange']

X_test = df_test[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
y_test = df_test['log2FoldChange']

# Train model
rf_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred = rf_model.predict(X_test)

# Evaluate
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n🔬 Performance on Genes Not in Training Set:")
print(f"R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")

# Show example predictions
df_test_results = df_test.copy()
df_test_results['Predicted_log2FoldChange'] = y_pred
df_test_results['Absolute_Error'] = abs(df_test_results['log2FoldChange'] - y_pred)

print("\n📋 Example Predictions on Unseen Genes:")
print(df_test_results[['Gene', 'TF', 'log2FoldChange', 'Predicted_log2FoldChange', 'Absolute_Error']].head(10))


In [ ]:
df_model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# ── Load & Encode ─────────────────────────────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene','TF','Tissue','log2FoldChange','TF_Family']].dropna()

# Label encoding for Gene, TF, Tissue
le_gene   = LabelEncoder().fit(df_model['Gene'])
le_tf     = LabelEncoder().fit(df_model['TF'])
le_tissue = LabelEncoder().fit(df_model['Tissue'])

df_model['Gene_encoded']   = le_gene.transform(df_model['Gene'])
df_model['TF_encoded']     = le_tf.transform(df_model['TF'])
df_model['Tissue_encoded'] = le_tissue.transform(df_model['Tissue'])

# Save example input
df_model.head(50).to_csv("example_input_dataframe.csv", index=False)

# ── Regression with TF & Family Target-Encoding ──────────────────────────────────
num_iterations = 10
sample_size    = min(20000, len(df_model))

cv_r2_scores        = []
cv_mse_scores       = []
test_r2_scores      = []
test_mse_scores     = []
feature_importances = []

final_predictions_df = None
final_training_df    = None

for i in range(num_iterations):
    print(f"\n🔁 Iteration {i+1}/{num_iterations}")
    df_sample = df_model.sample(n=sample_size, random_state=i).copy()

    # Target-encode TF and TF_Family by mean log2FoldChange
    df_sample['TF_te']        = df_sample.groupby('TF')['log2FoldChange'].transform('mean')
    df_sample['TF_Family_te'] = df_sample.groupby('TF_Family')['log2FoldChange'].transform('mean')

    # Prepare features & target
    X = df_sample[['Gene_encoded','TF_te','TF_Family_te','Tissue_encoded']]
    y = df_sample['log2FoldChange']

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i
    )

    # Mark categorical columns
    X_train['Gene_encoded']   = X_train['Gene_encoded'].astype('category')
    X_train['Tissue_encoded'] = X_train['Tissue_encoded'].astype('category')
    X_test['Gene_encoded']    = X_test['Gene_encoded'].astype('category')
    X_test['Tissue_encoded']  = X_test['Tissue_encoded'].astype('category')

    # Fit RandomForest
    rf_model = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=i)
    kf       = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv  = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')
    cv_r2_scores.append(r2_cv.mean())
    cv_mse_scores.append(mse_cv.mean())

    # Train on full train set, evaluate on test set
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    test_r2_scores.append(r2_score(y_test, y_pred))
    test_mse_scores.append(mean_squared_error(y_test, y_pred))

    # Record importances
    feature_importances.append(rf_model.feature_importances_)

    # On last iteration, save detailed outputs
    if i == num_iterations - 1:
        tf_family_map = df_model[['TF','TF_Family']].drop_duplicates()

        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange']    = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred
        X_test_copy['Gene']   = le_gene.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF']     = df_sample.loc[X_test_copy.index, 'TF']
        X_test_copy['Tissue'] = le_tissue.inverse_transform(X_test_copy['Tissue_encoded'])
        final_predictions_df = X_test_copy.merge(tf_family_map, on='TF', how='left')

        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene']   = le_gene.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF']     = df_sample.loc[X_train_copy.index, 'TF']
        X_train_copy['Tissue'] = le_tissue.inverse_transform(X_train_copy['Tissue_encoded'])
        final_training_df = X_train_copy.merge(tf_family_map, on='TF', how='left')

# ── Summary ────────────────────────────────────────────────────────────────────────
print("\n📈 Summary over all iterations:")
print(f"CV R²:    {np.mean(cv_r2_scores):.4f} ± {np.std(cv_r2_scores):.4f}")
print(f"CV MSE:   {np.mean(cv_mse_scores):.4f} ± {np.std(cv_mse_scores):.4f}")
print(f"Test R²:  {np.mean(test_r2_scores):.4f} ± {np.std(test_r2_scores):.4f}")
print(f"Test MSE: {np.mean(test_mse_scores):.4f} ± {np.std(test_mse_scores):.4f}")

# Save CSVs from final iteration
if final_predictions_df is not None:
    final_predictions_df.to_csv("random_forest_predictions_with_tf_family_te.csv", index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family_te.csv")
if final_training_df is not None:
    final_training_df.to_csv("random_forest_training_set_with_tf_family_te.csv", index=False)
    print("✅ Saved: random_forest_training_set_with_tf_family_te.csv")

# ── Visualizations ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(y=cv_r2_scores, ax=axes[0]).set_title("CV R² distribution")
sns.boxplot(y=test_r2_scores, ax=axes[1]).set_title("Test R² distribution")
plt.tight_layout()
plt.show()

avg_imp = np.mean(feature_importances, axis=0)
feat_names = ['Gene_encoded','TF_te','TF_Family_te','Tissue_encoded']
plt.figure(figsize=(6, 4))
sns.barplot(x=avg_imp, y=feat_names)
plt.title("Consensus Feature Importance")
plt.xlabel("Average Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd

# load & encode as before...
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df = df[['Gene','TF','Tissue','log2FoldChange']].dropna()
for col in ['Gene','TF','Tissue']:
    df[col+'_enc'] = pd.factorize(df[col])[0]

X = df[['Gene_enc','TF_enc','Tissue_enc']]
y = df['log2FoldChange']

def ablation_score(Xtrain, Xtest, ytrain, ytest, drop_col):
    rf_full = RandomForestRegressor(n_estimators=50, random_state=0)
    rf_full.fit(Xtrain, ytrain)
    full_r2 = r2_score(ytest, rf_full.predict(Xtest))

    Xtr_abl = Xtrain.drop(columns=[drop_col])
    Xte_abl = Xtest .drop(columns=[drop_col])
    rf_abl = RandomForestRegressor(n_estimators=50, random_state=0)
    rf_abl.fit(Xtr_abl, ytrain)
    abl_r2 = r2_score(ytest, rf_abl.predict(Xte_abl))

    return full_r2, abl_r2

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
full, abl = ablation_score(Xtr, Xte, ytr, yte, 'TF_enc')
print(f"R² full = {full:.3f}, R² w/o TF = {abl:.3f}  →  Δ = {full - abl:.3f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns

# ── Prep data ────────────────────────────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df = df[['Gene','TF','Tissue','log2FoldChange']].dropna()

# encode
le_g = LabelEncoder().fit(df['Gene'])
le_t = LabelEncoder().fit(df['TF'])
le_i = LabelEncoder().fit(df['Tissue'])
df['Gene_enc']   = le_g.transform(df['Gene'])
df['TF_enc']     = le_t.transform(df['TF'])
df['Tissue_enc'] = le_i.transform(df['Tissue'])

# ── Pipeline ─────────────────────────────────────────────────────────────────
n_iter      = 10
sample_size = min(20000, len(df))
perm_scores = []
drop_deltas = []

for i in range(n_iter):
    # 1) sample + split
    sub = df.sample(n=sample_size, random_state=i)
    X = sub[['Gene_enc','TF_enc','Tissue_enc']]
    y = sub['log2FoldChange']
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=i)
    for c in Xtr.columns:
        Xtr[c] = Xtr[c].astype('category')
        Xte[c] = Xte[c].astype('category')

    # 2) full model
    rf_full = RandomForestRegressor(n_estimators=50, random_state=i)
    rf_full.fit(Xtr, ytr)

    # 3) permutation importance on TF_enc
    perm = permutation_importance(
        rf_full, Xte, yte,
        scoring='r2', n_repeats=5, random_state=i
    )
    tf_imp = perm.importances_mean[list(Xte.columns).index('TF_enc')]
    perm_scores.append(tf_imp)

    # 4) ablation: train on Gene+Tissue only
    rf_abl = RandomForestRegressor(n_estimators=50, random_state=i)
    rf_abl.fit(Xtr.drop('TF_enc',axis=1), ytr)

    # 5) R² drop
    r2_full = r2_score(yte, rf_full.predict(Xte))
    r2_abl  = r2_score(yte, rf_abl.predict(Xte.drop('TF_enc',axis=1)))
    drop_deltas.append(r2_full - r2_abl)

# ── Summarize ────────────────────────────────────────────────────────────────
print(f"Permutation ΔR² for TF  : {np.mean(perm_scores):.4f} ± {np.std(perm_scores):.4f}")
print(f"Ablation ΔR² (full–noTF): {np.mean(drop_deltas):.4f} ± {np.std(drop_deltas):.4f}")

# ── Visualize ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1,2, figsize=(10,4))
sns.boxplot(y=perm_scores, ax=ax[0]).set_title("Perm ΔR² (TF_enc)")
sns.boxplot(y=drop_deltas,ax=ax[1]).set_title("Drop-in ΔR² (full–noTF)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# 1) Load & label-encode once on the full data
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df = df[['Gene', 'TF', 'Tissue', 'log2FoldChange']].dropna()

le_gene   = LabelEncoder().fit(df['Gene'])
le_tf     = LabelEncoder().fit(df['TF'])
le_tissue = LabelEncoder().fit(df['Tissue'])

df['Gene_enc']   = le_gene.transform(df['Gene'])
df['TF_enc']     = le_tf.transform(df['TF'])
df['Tissue_enc'] = le_tissue.transform(df['Tissue'])

# 2) Prepare X/y
X = df[['Gene_enc','TF_enc','Tissue_enc']]
y = df['log2FoldChange']

# 3) Run 10× 20k-sample RF experiments
N_ITERS = 10
SAMP    = min(20000, len(df))

cv_r2, cv_mse, test_r2, test_mse = [], [], [], []

for seed in range(N_ITERS):
    # a) sample + split
    idx = df.sample(n=SAMP, random_state=seed).index
    Xs, ys = X.loc[idx], y.loc[idx]
    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=seed)
    
    # b) cross-val
    rf = RandomForestRegressor(n_estimators=50, random_state=seed, n_jobs=-1)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    cv_r2.append( cross_val_score(rf, Xtr, ytr, cv=kf, scoring='r2').mean() )
    cv_mse.append(-cross_val_score(rf, Xtr, ytr, cv=kf, scoring='neg_mean_squared_error').mean())
    
    # c) fit + test
    rf.fit(Xtr, ytr)
    preds = rf.predict(Xte)
    test_r2.append( r2_score(yte, preds) )
    test_mse.append( mean_squared_error(yte, preds) )

# 4) Summaries
print(f"CV   R²: {np.mean(cv_r2):.4f} ± {np.std(cv_r2):.4f}")
print(f"CV  MSE: {np.mean(cv_mse):.4f} ± {np.std(cv_mse):.4f}")
print(f"Test R²: {np.mean(test_r2):.4f} ± {np.std(test_r2):.4f}")
print(f"Test MSE:{np.mean(test_mse):.4f} ± {np.std(test_mse):.4f}")

# 5) Boxplot of Test R²
plt.figure(figsize=(6,4))
sns.boxplot(data=test_r2, orient='v')
plt.ylabel("Test R²")
plt.title("Distribution of Test R² over 10 runs")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

# ── 1) Load & Label-Encode ─────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df = df[['Gene','TF','Tissue','log2FoldChange']].dropna().copy()

le_gene   = LabelEncoder().fit(df['Gene'])
le_tf     = LabelEncoder().fit(df['TF'])
le_tissue = LabelEncoder().fit(df['Tissue'])

df['Gene_enc']   = le_gene.transform(df['Gene'])
df['TF_enc']     = le_tf.transform(df['TF'])
df['Tissue_enc'] = le_tissue.transform(df['Tissue'])

# ── 2) Prep storage for our 10 runs ───────────────────────────────
n_iters     = 10
perm_deltas = []   # permutation ΔR² for TF
abl_deltas  = []   # ablation ΔR² = R²_full − R²_noTF

# ── 3) Loop over random splits ────────────────────────────────────
for seed in range(n_iters):
    # 3a) sample & split
    sample = df.sample(n=20000, random_state=seed)
    X = sample[['Gene_enc','TF_enc','Tissue_enc']]
    y = sample['log2FoldChange']
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)

    # 3b) Train full RF
    rf_full = RandomForestRegressor(n_estimators=50, random_state=seed, n_jobs=-1)
    rf_full.fit(Xtr, ytr)
    r2_full = r2_score(yte, rf_full.predict(Xte))

    # 3c) Permutation importance on TF_enc
    perm = permutation_importance(rf_full, Xte, yte,
                                  n_repeats=30,
                                  random_state=seed,
                                  scoring='r2',
                                  n_jobs=-1)
    tf_idx = list(Xte.columns).index('TF_enc')
    perm_deltas.append(perm.importances_mean[tf_idx])

    # 3d) Ablation: drop TF_enc, retrain & retest
    rf_noTF = RandomForestRegressor(n_estimators=50, random_state=seed, n_jobs=-1)
    rf_noTF.fit(Xtr.drop('TF_enc', axis=1), ytr)
    r2_noTF = r2_score(
        yte,
        rf_noTF.predict(Xte.drop('TF_enc', axis=1))
    )
    abl_deltas.append(r2_full - r2_noTF)

# ── 4) Final summary ───────────────────────────────────────────────
print(f"Permutation ΔR² for TF : {np.mean(perm_deltas):.4f} ± {np.std(perm_deltas):.4f}")
print(f"Ablation ΔR² (full−noTF): {np.mean(abl_deltas):.4f} ± {np.std(abl_deltas):.4f}")


In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

# ── 1) Load & Label-encode, cast to categorical ───────────────────────────────
df = (
    pd.read_csv("final_filtered_gene_tf_data.csv")
      .loc[:, ['Gene','TF','Tissue','log2FoldChange']]
      .dropna()
)

# label encode each
for col in ['Gene','TF','Tissue']:
    df[f"{col}_enc"] = pd.Categorical(df[col]).codes
    # now make the encoded column a pandas Categorical
    df[f"{col}_enc"] = df[f"{col}_enc"].astype("category")

# ── 2) Helper: permute one column within groups ─────────────────────────────────
def permute_within_group(X_df, feature, by, random_state=None):
    Xp = X_df.copy()
    rng = np.random.default_rng(random_state)
    Xp[feature] = (
        Xp.groupby(by)[feature]
           .transform(lambda col: rng.permutation(col.values))
    )
    return Xp

# ── 3) Run n_iters of splits + analyses ───────────────────────────────────────
n_iters = 10
results = []

for seed in range(n_iters):
    # a) sample & split
    samp = df.sample(n=20000, random_state=seed)
    X = samp[['Gene_enc','TF_enc','Tissue_enc']]
    y = samp['log2FoldChange']
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )

    # b) full RF
    rf_full = RandomForestRegressor(
        n_estimators=100, random_state=seed, n_jobs=-1
    )
    rf_full.fit(Xtr, ytr)
    r2_full = r2_score(yte, rf_full.predict(Xte))

    # c) global permutation importance on TF_enc
    perm = permutation_importance(
        rf_full, Xte, yte,
        scoring='r2', n_repeats=30,
        random_state=seed, n_jobs=-1
    )
    tf_idx = list(Xte.columns).index('TF_enc')
    perm_tf = perm.importances_mean[tf_idx]

    # d) permute TF within gene
    Xte_df = Xte.reset_index(drop=True)
    Xte_wg = permute_within_group(Xte_df, 'TF_enc', by='Gene_enc', random_state=seed)
    r2_wg = r2_score(yte, rf_full.predict(Xte_wg))

    # e) TF-only model
    rf_tf = RandomForestRegressor(
        n_estimators=100, random_state=seed, n_jobs=-1
    )
    rf_tf.fit(Xtr[['TF_enc']], ytr)
    r2_tf_only = r2_score(yte, rf_tf.predict(Xte[['TF_enc']]))

    # f) (optional) ablation drop TF_enc
    rf_noTF = RandomForestRegressor(
        n_estimators=100, random_state=seed, n_jobs=-1
    )
    rf_noTF.fit(Xtr.drop('TF_enc', axis=1), ytr)
    r2_noTF = r2_score(
        yte,
        rf_noTF.predict(Xte.drop('TF_enc', axis=1))
    )
    abl = r2_full - r2_noTF

    # record
    results.append({
        'seed': seed,
        'r2_full':      r2_full,
        'perm_tf':      perm_tf,
        'r2_within_g':  r2_full - r2_wg,
        'r2_tf_only':   r2_tf_only,
        'ablation_tf':  abl
    })

res = pd.DataFrame(results)
print("\nMean ± std over runs:")
print(res[['r2_full','perm_tf','r2_within_g','r2_tf_only','ablation_tf']].agg(['mean','std']))


In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── 1) Load & Label-Encode ─────────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df = df[['Gene','TF','Tissue','log2FoldChange']].dropna().copy()

# simple label-encoders
le_gene   = LabelEncoder().fit(df['Gene'])
le_tf     = LabelEncoder().fit(df['TF'])
le_tissue = LabelEncoder().fit(df['Tissue'])

df['Gene_enc']   = le_gene.transform(df['Gene'])
df['TF_enc']     = le_tf.transform(df['TF'])
df['Tissue_enc'] = le_tissue.transform(df['Tissue'])

# composite feature
df['Gene_TF'] = df['Gene'] + "_" + df['TF']
le_gene_tf   = LabelEncoder().fit(df['Gene_TF'])
df['Gene_TF_enc'] = le_gene_tf.transform(df['Gene_TF'])


# ── 2) Define the various feature-sets to compare ─────────────────────
feature_sets = {
    'gene_only'       : ['Gene_enc'],
    'gene_plus_TF'    : ['Gene_enc','TF_enc'],
    'composite_only'  : ['Gene_TF_enc'],
    'composite+tissue': ['Gene_TF_enc','Tissue_enc'],
    'full'            : ['Gene_enc','TF_enc','Tissue_enc']
}

# storage for repeated runs
n_iters = 10
results = {name: [] for name in feature_sets}


# ── 3) Loop: sample, split, train & test ───────────────────────────────
for seed in range(n_iters):
    data = df.sample(n=20000, random_state=seed)
    X = data
    y = data['log2FoldChange']
    
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    for name, feats in feature_sets.items():
        # select & cast to categorical
        Xtr_sub = Xtr[feats].copy()
        Xte_sub = Xte[feats].copy()
        for c in feats:
            Xtr_sub[c] = Xtr_sub[c].astype('category')
            Xte_sub[c] = Xte_sub[c].astype('category')
        
        # train
        rf = RandomForestRegressor(
            n_estimators=50, random_state=seed, n_jobs=-1
        )
        rf.fit(Xtr_sub, ytr)
        r2 = r2_score(yte, rf.predict(Xte_sub))
        results[name].append(r2)


# ── 4) Summarize & plot ────────────────────────────────────────────────
# a) numeric summary
summary = (
    pd.DataFrame(results)
      .agg(['mean','std'])
      .T
      .rename(columns={'mean':'R²_mean','std':'R²_std'})
)
print("\nPerformance across feature-sets:\n")
print(summary)

# b) boxplot of R² distributions
plt.figure(figsize=(8,4))
sns.boxplot(data=pd.DataFrame(results))
plt.ylabel("Test R²")
plt.title("How much each feature-set explains log₂FC")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── 1) Load & Label-Encode ────────────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df = df[['Gene','TF','Tissue','log2FoldChange']].dropna().copy()

le_gene   = LabelEncoder().fit(df['Gene'])
le_tf     = LabelEncoder().fit(df['TF'])
le_tissue = LabelEncoder().fit(df['Tissue'])

df['Gene_enc']   = le_gene.transform(df['Gene'])
df['TF_enc']     = le_tf.transform(df['TF'])
df['Tissue_enc'] = le_tissue.transform(df['Tissue'])

# Composite feature
df['Gene_TF'] = df['Gene'] + "_" + df['TF']
le_gene_tf   = LabelEncoder().fit(df['Gene_TF'])
df['Gene_TF_enc'] = le_gene_tf.transform(df['Gene_TF'])


# ── 2) Define feature-sets ────────────────────────────────────────────────
feature_sets = {
    'gene_only'       : ['Gene_enc'],
    'gene_plus_TF'    : ['Gene_enc','TF_enc'],
    'composite_only'  : ['Gene_TF_enc'],
    'composite+tissue': ['Gene_TF_enc','Tissue_enc'],
    'full'            : ['Gene_enc','TF_enc','Tissue_enc']
}

# ── 3) Repeat train/test & collect R² ────────────────────────────────────
n_iters = 10
results = {name: [] for name in feature_sets}

for seed in range(n_iters):
    data = df.sample(n=20000, random_state=seed)
    X = data;  y = data['log2FoldChange']
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    for name, feats in feature_sets.items():
        Xtr_sub = Xtr[feats].astype('category')
        Xte_sub = Xte[feats].astype('category')
        rf = RandomForestRegressor(n_estimators=50, random_state=seed, n_jobs=-1)
        rf.fit(Xtr_sub, ytr)
        results[name].append(r2_score(yte, rf.predict(Xte_sub)))


# ── 4a) Numeric summary ──────────────────────────────────────────────────
summary = (
    pd.DataFrame(results)
      .agg(['mean','std'])
      .T
      .rename(columns={'mean':'R²_mean','std':'R²_std'})
)
print("\nPerformance across feature-sets:\n")
print(summary)


# ── 4b) Nicer box-plot ────────────────────────────────────────────────────
# 1) Rename for readability
rename_map = {
    'gene_only'       : 'Gene only',
    'gene_plus_TF'    : 'Gene + TF',
    'composite_only'  : 'Composite only',
    'composite+tissue': 'Composite + Tissue',
    'full'            : 'Full'
}
df_plot = pd.DataFrame(results).rename(columns=rename_map)

# 2) Determine order by mean R²
order_orig = summary.sort_values('R²_mean').index.tolist()
order_plot = [rename_map[o] for o in order_orig]

# 3) Set theme & draw
plt.figure(figsize=(10,6))
sns.set_theme(style='whitegrid')
sns.boxplot(data=df_plot, palette='pastel', order=order_plot)
sns.stripplot(data=df_plot, color='gray', size=3, jitter=True, order=order_plot)

# 4) Annotate medians
for i, col in enumerate(order_plot):
    med = df_plot[col].median()
    plt.text(i, med + 0.005, f"{med:.2f}",
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# 5) Labels & styling
plt.ylabel("Test $R^2$", fontsize=12)
plt.title("Feature-set Comparison: Test $R^2$ Distribution", fontsize=14)
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)
plt.ylim(df_plot.min().min() - 0.05, df_plot.max().max() + 0.05)
sns.despine(trim=True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────────────────────────────────────
# 0) Load & prep
# ──────────────────────────────────────────────────────────────────────────────
csv_path = "final_filtered_gene_tf_data.csv"
df = (
    pd.read_csv(csv_path)[['Gene','TF','Tissue','log2FoldChange']]
    .dropna()
    .copy()
)

# mark Tissue as categorical
df['Tissue'] = df['Tissue'].astype('category')

# composite ID for random effects
df['composite'] = df['Gene'] + "_" + df['TF']


# ──────────────────────────────────────────────────────────────────────────────
# 1) Mixed‐effects model: random intercept + random slope of Tissue per composite
# ──────────────────────────────────────────────────────────────────────────────
model = smf.mixedlm(
    "log2FoldChange ~ C(Tissue)",
    df,
    groups=df["composite"],
    re_formula="~C(Tissue)"
)
mixed_res = model.fit(method='lbfgs')
print("\n── MixedLM summary ──")
print(mixed_res.summary())



In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# OUTPUT DIRECTORY (same as before)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Current working directory:", os.getcwd())
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))


def save_figure(base_name, dpi=300):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")
    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()
    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)


# ============================================================
# Load dataset
# ============================================================
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[["Gene", "TF", "Tissue", "log2FoldChange", "TF_Family"]].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model["Gene_encoded"] = gene_le.fit_transform(df_model["Gene"].astype(str))
df_model["TF_encoded"] = tf_le.fit_transform(df_model["TF"].astype(str))
df_model["Tissue_encoded"] = tissue_le.fit_transform(df_model["Tissue"].astype(str))

# ============================================================
# Save example input dataframe (TOP 50) into OUTPUT_DIR
# ============================================================
example_path = os.path.join(OUTPUT_DIR, "example_input_dataframe.csv")
df_model.head(50).to_csv(example_path, index=False)
print("✅ Saved:", example_path)

# ============================================================
# Trackers
# ============================================================
num_iterations = 10
target_n = 20000
rows_used = min(target_n, len(df_model))

if rows_used < target_n:
    print(f"⚠️ Dataset has only {len(df_model)} rows after dropna; using n={rows_used} per iteration instead of 20000.")

cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

final_predictions_df = None
final_training_df = None

# Build TF_Family lookup once (stable)
tf_family_map = df_model[["TF", "TF_Family"]].drop_duplicates()

# ============================================================
# Loop
# ============================================================
for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")

    df_sample = df_model.sample(n=rows_used, random_state=i)

    X = df_sample[["Gene_encoded", "TF_encoded", "Tissue_encoded"]]
    y = df_sample["log2FoldChange"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i
    )

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i, n_jobs=-1)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring="r2")
    mse_cv = -cross_val_score(
        rf_model, X_train, y_train, cv=kf, scoring="neg_mean_squared_error"
    )

    cv_r2_scores.append(float(np.mean(r2_cv)))
    cv_mse_scores.append(float(np.mean(mse_cv)))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(float(r2_test))
    test_mse_scores.append(float(mse_test))

    # Save outputs from final iteration
    if i == num_iterations - 1:
        # Test predictions with decoded labels
        X_test_copy = X_test.copy()
        X_test_copy["Actual_log2FoldChange"] = y_test.values
        X_test_copy["Predicted_log2FoldChange"] = y_pred_test

        X_test_copy["Gene"] = gene_le.inverse_transform(X_test_copy["Gene_encoded"].astype(int))
        X_test_copy["TF"] = tf_le.inverse_transform(X_test_copy["TF_encoded"].astype(int))
        X_test_copy["Tissue"] = tissue_le.inverse_transform(X_test_copy["Tissue_encoded"].astype(int))

        X_test_copy = X_test_copy.merge(tf_family_map, on="TF", how="left")
        final_predictions_df = X_test_copy

        # Training set with decoded labels
        X_train_copy = X_train.copy()
        X_train_copy["log2FoldChange"] = y_train.values

        X_train_copy["Gene"] = gene_le.inverse_transform(X_train_copy["Gene_encoded"].astype(int))
        X_train_copy["TF"] = tf_le.inverse_transform(X_train_copy["TF_encoded"].astype(int))
        X_train_copy["Tissue"] = tissue_le.inverse_transform(X_train_copy["Tissue_encoded"].astype(int))

        X_train_copy = X_train_copy.merge(tf_family_map, on="TF", how="left")
        final_training_df = X_train_copy


# ============================================================
# Summary
# ============================================================
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# ============================================================
# Save outputs into OUTPUT_DIR
# ============================================================
if final_predictions_df is not None:
    pred_path = os.path.join(OUTPUT_DIR, "random_forest_predictions_with_tf_family.csv")
    final_predictions_df.to_csv(pred_path, index=False)
    print("✅ Saved:", pred_path)

if final_training_df is not None:
    train_path = os.path.join(OUTPUT_DIR, "random_forest_training_set.csv")
    final_training_df.to_csv(train_path, index=False)
    print("✅ Saved:", train_path)

# Also save iteration metrics (super useful later)
iter_metrics = pd.DataFrame({
    "iteration": np.arange(1, num_iterations + 1),
    "cv_r2": cv_r2_scores,
    "cv_mse": cv_mse_scores,
    "test_r2": test_r2_scores,
    "test_mse": test_mse_scores
})
iter_metrics_path = os.path.join(OUTPUT_DIR, "rf_iteration_metrics.csv")
iter_metrics.to_csv(iter_metrics_path, index=False)
print("✅ Saved:", iter_metrics_path)

# ============================================================
# Boxplots (save PNG + SVG)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.boxplot(y=cv_r2_scores, ax=axes[0])
axes[0].set_title("Cross-Validation R² Distribution")

sns.boxplot(y=test_r2_scores, ax=axes[1])
axes[1].set_title("Test R² Distribution")

save_figure("rf_r2_distributions_with_iterations")

print("\n🎉 Done. All outputs are in:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
# ── Updated 4b) Nicer box-plot with values inside bars ──────────────────────
# 1) Rename for readability
rename_map = {
    'gene_only'       : 'Gene only',
    'gene_plus_TF'    : 'Gene + TF',
    'composite_only'  : 'Composite only',
    'composite+tissue': 'Composite + Tissue',
    'full'            : 'Full'
}
df_plot = pd.DataFrame(results).rename(columns=rename_map)

# 2) Determine order by mean R²
order_orig = summary.sort_values('R²_mean').index.tolist()
order_plot = [rename_map[o] for o in order_orig]

# 3) Set theme & draw
plt.figure(figsize=(10,6))
sns.set_theme(style='whitegrid')
sns.boxplot(data=df_plot, palette='pastel', order=order_plot)
sns.stripplot(data=df_plot, color='gray', size=3, jitter=True, order=order_plot)

# 4) Annotate median values inside each box
medians = df_plot.median()
for i, col in enumerate(order_plot):
    median_val = medians[col]
    plt.text(
        i, median_val, f"{median_val:.2f}",
        ha='center', va='center',
        fontsize=9, fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2')
    )

# 5) Labels & styling
plt.ylabel("Test $R^2$", fontsize=12)
plt.title("Feature-set Comparison: Test $R^2$ Distribution", fontsize=14)
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)
plt.ylim(df_plot.min().min() - 0.05, df_plot.max().max() + 0.05)
sns.despine(trim=True)
plt.tight_layout()
plt.show()


In [ ]:
# ── Nicer box-plot with values inside bars + save to specific folder ─────────
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 0) Output directory (your provided path)
# ------------------------------------------------------------
outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

# ------------------------------------------------------------
# 1) Rename for readability
# ------------------------------------------------------------
rename_map = {
    'gene_only'       : 'Gene only',
    'gene_plus_TF'    : 'Gene + TF',
    'composite_only'  : 'Composite only',
    'composite+tissue': 'Composite + Tissue',
    'full'            : 'Full'
}

df_plot = pd.DataFrame(results).rename(columns=rename_map)

# ------------------------------------------------------------
# 2) Determine order by mean R²
# ------------------------------------------------------------
order_orig = summary.sort_values('R²_mean').index.tolist()
order_plot = [rename_map[o] for o in order_orig]

# ------------------------------------------------------------
# 3) Set theme & draw
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.set_theme(style='whitegrid')

sns.boxplot(data=df_plot, palette='pastel', order=order_plot)
sns.stripplot(data=df_plot, color='gray', size=3, jitter=True, order=order_plot)

# ------------------------------------------------------------
# 4) Annotate median values inside each box
# ------------------------------------------------------------
medians = df_plot.median(numeric_only=True)

for i, col in enumerate(order_plot):
    median_val = float(medians[col])
    plt.text(
        i, median_val, f"{median_val:.2f}",
        ha='center', va='center',
        fontsize=9, fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2')
    )

# ------------------------------------------------------------
# 5) Labels & styling
# ------------------------------------------------------------
plt.ylabel("Test $R^2$", fontsize=12)
plt.title("Feature-set Comparison: Test $R^2$ Distribution", fontsize=14)
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)

plt.ylim(
    df_plot.min(numeric_only=True).min() - 0.05,
    df_plot.max(numeric_only=True).max() + 0.05
)

sns.despine(trim=True)
plt.tight_layout()

# ------------------------------------------------------------
# 6) Save outputs
# ------------------------------------------------------------
svg_path = os.path.join(outdir, "feature_set_comparison_r2.svg")
png_path = os.path.join(outdir, "feature_set_comparison_r2.png")

plt.savefig(svg_path, format="svg", bbox_inches="tight")
plt.savefig(png_path, dpi=300, bbox_inches="tight")

print(f"Saved SVG → {svg_path}")
print(f"Saved PNG → {png_path}")

plt.show()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# OUTPUT DIRECTORY (same as before)
# ============================================================
OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("📁 All outputs will be saved in:", os.path.abspath(OUTPUT_DIR))

def save_figure(base_name, dpi=300):
    png_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    svg_path = os.path.join(OUTPUT_DIR, f"{base_name}.svg")
    plt.tight_layout()
    plt.savefig(png_path, dpi=dpi)
    plt.savefig(svg_path)
    plt.close()
    print("✅ Saved figure:", png_path)
    print("✅ Saved figure:", svg_path)

# ============================================================
# REQUIREMENTS CHECK
# ============================================================
missing = [name for name in ["results", "summary"] if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing required variables in memory: "
        + ", ".join(missing)
        + ". Make sure you ran the feature-set comparison code that defines them."
    )

# ── Updated 4b) Nicer box-plot with values inside bars ──────────────────────
# 1) Rename for readability
rename_map = {
    "gene_only": "Gene only",
    "gene_plus_TF": "Gene + TF",
    "composite_only": "Composite only",
    "composite+tissue": "Composite + Tissue",
    "full": "Full",
}

df_plot = pd.DataFrame(results).rename(columns=rename_map)

# 2) Determine order by mean R²
# Assumes summary has a column named 'R²_mean' and index with the original keys.
order_orig = summary.sort_values("R²_mean").index.tolist()
order_plot = [rename_map[o] for o in order_orig if o in rename_map]  # safe mapping

# 3) Set theme & draw
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
sns.boxplot(data=df_plot, order=order_plot, palette="pastel")
sns.stripplot(data=df_plot, order=order_plot, color="gray", size=3, jitter=True)

# 4) Annotate median values inside each box
medians = df_plot.median(numeric_only=True)
for i, col in enumerate(order_plot):
    if col not in medians:
        continue
    median_val = float(medians[col])
    plt.text(
        i,
        median_val,
        f"{median_val:.2f}",
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.7, boxstyle="round,pad=0.2"),
    )

# 5) Labels & styling
plt.ylabel("Test $R^2$", fontsize=12)
plt.title("Feature-set Comparison: Test $R^2$ Distribution", fontsize=14)
plt.xticks(rotation=45, fontsize=10)
plt.yticks(fontsize=10)

ymin = df_plot.min(numeric_only=True).min()
ymax = df_plot.max(numeric_only=True).max()
plt.ylim(ymin - 0.05, ymax + 0.05)

sns.despine(trim=True)

# SAVE (PNG + SVG) into analysis_outputs
save_figure("feature_set_comparison_test_r2_boxplot_with_medians")

print("\n🎉 Done. Plot saved in:")
print(os.path.abspath(OUTPUT_DIR))


In [ ]:
# -*- coding: utf-8 -*-
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt
import seaborn as sns

# ───────────────────────────────────────────────────────────────────────────────
# Utility: leakage-safe target encoding
# Produces: OOF means for train; test means built only from train folds.
# Handles unseen categories in test by global mean fallback.
# ───────────────────────────────────────────────────────────────────────────────
def oof_target_encode(train_df, test_df, cat_col, y_col, n_splits=5, seed=0):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = pd.Series(np.nan, index=train_df.index, dtype=np.float32)

    global_mean = train_df[y_col].mean()

    for tr_idx, val_idx in kf.split(train_df):
        tr_part = train_df.iloc[tr_idx]
        val_part = train_df.iloc[val_idx]
        means = tr_part.groupby(cat_col)[y_col].mean().astype(np.float32)
        oof.iloc[val_idx] = val_part[cat_col].map(means).fillna(global_mean).values

    # test transform (strictly from full train)
    full_means = train_df.groupby(cat_col)[y_col].mean().astype(np.float32)
    test_te = test_df[cat_col].map(full_means).fillna(global_mean).astype(np.float32)

    return oof.astype(np.float32), test_te

# ───────────────────────────────────────────────────────────────────────────────
# Load data
# ───────────────────────────────────────────────────────────────────────────────
INPUT_CSV = "final_filtered_gene_tf_data.csv"
df0 = (
    pd.read_csv(INPUT_CSV)
      .loc[:, ["Gene","TF","Tissue","log2FoldChange","TF_Family"]]
      .dropna()
      .reset_index(drop=True)
)

# Save a small preview for reference
df0.head(50).to_csv("example_input_dataframe.csv", index=False)

# Label-encode categories → numeric codes (int32)
le_gene   = LabelEncoder().fit(df0["Gene"])
le_tf     = LabelEncoder().fit(df0["TF"])
le_tissue = LabelEncoder().fit(df0["Tissue"])

df0["Gene_enc"]   = le_gene.transform(df0["Gene"]).astype(np.int32)
df0["TF_enc"]     = le_tf.transform(df0["TF"]).astype(np.int32)
df0["Tissue_enc"] = le_tissue.transform(df0["Tissue"]).astype(np.int32)

# ───────────────────────────────────────────────────────────────────────────────
# Experiment settings
# ───────────────────────────────────────────────────────────────────────────────
N_ITERS     = 10
MAX_SAMP    = min(20000, len(df0))
N_ESTIMATORS= 100

cv_r2s, cv_mses, te_r2s, te_mses = [], [], [], []
perm_tf_deltas, abl_deltas = [], []
feat_importances_runs = []

final_predictions_df = None
final_training_df    = None

for seed in range(N_ITERS):
    print(f"\n🔁 Iteration {seed+1}/{N_ITERS}")

    # Sample for this run (keeps runtime bounded & comparable)
    sub = df0.sample(n=MAX_SAMP, random_state=seed).reset_index(drop=True)

    # Split
    X_base = sub[["Gene_enc","TF_enc","Tissue_enc"]].copy()
    y = sub["log2FoldChange"].astype(np.float32).copy()

    X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
        X_base, y, sub.index, test_size=0.2, random_state=seed
    )

    # Build DataFrames with original string cols for target-encoding
    tr_df = sub.loc[idx_tr, ["TF","TF_Family","log2FoldChange"]].copy()
    te_df = sub.loc[idx_te, ["TF","TF_Family","log2FoldChange"]].copy()

    # Leakage-safe target encodings for TF and TF_Family (OOF on train; train-only on test)
    tf_te_tr, tf_te_te = oof_target_encode(
        train_df=tr_df.rename(columns={"log2FoldChange":"y"}),
        test_df=te_df.rename(columns={"log2FoldChange":"y"}),
        cat_col="TF", y_col="y", n_splits=5, seed=seed
    )

    fam_te_tr, fam_te_te = oof_target_encode(
        train_df=tr_df.rename(columns={"log2FoldChange":"y"}).assign(TF_Family=tr_df["TF_Family"]),
        test_df=te_df.rename(columns={"log2FoldChange":"y"}).assign(TF_Family=te_df["TF_Family"]),
        cat_col="TF_Family", y_col="y", n_splits=5, seed=seed
    )

    # Assemble final feature matrices (float32)
    X_train = pd.DataFrame({
        "Gene_enc"     : X_tr["Gene_enc"].astype(np.int32),
        "Tissue_enc"   : X_tr["Tissue_enc"].astype(np.int32),
        "TF_te"        : tf_te_tr.values.astype(np.float32),
        "TF_Family_te" : fam_te_tr.values.astype(np.float32),
    })

    X_test = pd.DataFrame({
        "Gene_enc"     : X_te["Gene_enc"].astype(np.int32),
        "Tissue_enc"   : X_te["Tissue_enc"].astype(np.int32),
        "TF_te"        : tf_te_te.values.astype(np.float32),
        "TF_Family_te" : fam_te_te.values.astype(np.float32),
    })

    # Model + CV (on leakage-safe training matrix)
    rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS, random_state=seed, n_jobs=-1
    )
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    cv_r2  = cross_val_score(rf, X_train, y_tr, cv=kf, scoring="r2").mean()
    cv_mse = -cross_val_score(rf, X_train, y_tr, cv=kf, scoring="neg_mean_squared_error").mean()
    cv_r2s.append(cv_r2);  cv_mses.append(cv_mse)

    # Fit & test
    rf.fit(X_train, y_tr)
    y_pred = rf.predict(X_test)
    r2 = r2_score(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    te_r2s.append(r2); te_mses.append(mse)

    feat_importances_runs.append(rf.feature_importances_)

    # Permutation importance (focus on TF_te signal)
    perm = permutation_importance(
        rf, X_test, y_te, scoring="r2", n_repeats=10, random_state=seed, n_jobs=-1
    )
    tf_idx = list(X_test.columns).index("TF_te")
    perm_tf_deltas.append(float(perm.importances_mean[tf_idx]))

    # Ablation: drop TF_te and TF_Family_te, retrain on (Gene_enc, Tissue_enc) only
    rf_noTF = RandomForestRegressor(
        n_estimators=N_ESTIMATORS, random_state=seed, n_jobs=-1
    )
    rf_noTF.fit(X_train[["Gene_enc","Tissue_enc"]], y_tr)
    r2_noTF = r2_score(y_te, rf_noTF.predict(X_test[["Gene_enc","Tissue_enc"]]))
    abl_deltas.append(r2 - r2_noTF)

    # On final iteration, write detailed CSVs
    if seed == N_ITERS - 1:
        # reconstruct readable columns for test
        test_out = pd.DataFrame({
            "Gene_enc"              : X_test["Gene_enc"].values,
            "Tissue_enc"            : X_test["Tissue_enc"].values,
            "TF_te"                 : X_test["TF_te"].values,
            "TF_Family_te"          : X_test["TF_Family_te"].values,
            "Actual_log2FoldChange" : y_te.values,
            "Predicted_log2FoldChange": y_pred
        }, index=idx_te)

        # add back string labels
        test_out["Gene"]   = le_gene.inverse_transform(test_out["Gene_enc"].astype(int))
        test_out["Tissue"] = le_tissue.inverse_transform(test_out["Tissue_enc"].astype(int))
        # recover TF and TF_Family from the original sampled frame
        test_out["TF"]         = sub.loc[idx_te, "TF"].values
        test_out["TF_Family"]  = sub.loc[idx_te, "TF_Family"].values

        train_out = pd.DataFrame({
            "Gene_enc"     : X_train["Gene_enc"].values,
            "Tissue_enc"   : X_train["Tissue_enc"].values,
            "TF_te"        : X_train["TF_te"].values,
            "TF_Family_te" : X_train["TF_Family_te"].values,
            "log2FoldChange": y_tr.values
        }, index=idx_tr)
        train_out["Gene"]   = le_gene.inverse_transform(train_out["Gene_enc"].astype(int))
        train_out["Tissue"] = le_tissue.inverse_transform(train_out["Tissue_enc"].astype(int))
        train_out["TF"]         = sub.loc[idx_tr, "TF"].values
        train_out["TF_Family"]  = sub.loc[idx_tr, "TF_Family"].values

        final_predictions_df = test_out.reset_index(drop=True)
        final_training_df    = train_out.reset_index(drop=True)

# ───────────────────────────────────────────────────────────────────────────────
# Summary
# ───────────────────────────────────────────────────────────────────────────────
print("\n📈 Summary over all iterations:")
print(f"CV   R² : {np.mean(cv_r2s):.4f} ± {np.std(cv_r2s):.4f}")
print(f"CV  MSE : {np.mean(cv_mses):.4f} ± {np.std(cv_mses):.4f}")
print(f"Test R² : {np.mean(te_r2s):.4f} ± {np.std(te_r2s):.4f}")
print(f"Test MSE: {np.mean(te_mses):.4f} ± {np.std(te_mses):.4f}")
print(f"Perm ΔR² (TF_te): {np.mean(perm_tf_deltas):.4f} ± {np.std(perm_tf_deltas):.4f}")
print(f"Ablation ΔR² (full − no TF_te/Fam): {np.mean(abl_deltas):.4f} ± {np.std(abl_deltas):.4f}")

# Save CSVs from last run
if final_predictions_df is not None:
    final_predictions_df.to_csv("rf_predictions_leakage_safe_te.csv", index=False)
    print("✅ Saved: rf_predictions_leakage_safe_te.csv")
if final_training_df is not None:
    final_training_df.to_csv("rf_training_leakage_safe_te.csv", index=False)
    print("✅ Saved: rf_training_leakage_safe_te.csv")

# ───────────────────────────────────────────────────────────────────────────────
# Visualizations (vertical)
# ───────────────────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid")

# Boxplots: CV/Test R²
plt.figure(figsize=(4,6))
sns.boxplot(y=cv_r2s)
plt.title("CV R² distribution")
plt.ylabel("R²")
plt.tight_layout()
plt.show()

plt.figure(figsize=(4,6))
sns.boxplot(y=te_r2s)
plt.title("Test R² distribution")
plt.ylabel("R²")
plt.tight_layout()
plt.show()

# Feature importances (mean over runs)
avg_imp = np.mean(np.vstack(feat_importances_runs), axis=0)
feat_names = ["Gene_enc","Tissue_enc","TF_te","TF_Family_te"]
order = np.argsort(avg_imp)
plt.figure(figsize=(5,6))
sns.barplot(x=avg_imp[order], y=np.array(feat_names)[order], orient="h")
plt.xlabel("Average Importance")
plt.title("Consensus Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load dataset
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

# Label Encoding
gene_le = LabelEncoder()
tf_le = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded'] = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded'] = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# Save example input dataframe
df_model.head(50).to_csv("example_input_dataframe.csv", index=False)

# Trackers
num_iterations = 10
cv_r2_scores = []
cv_mse_scores = []
test_r2_scores = []
test_mse_scores = []

final_predictions_df = None
final_training_df = None

# Loop
for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=i)

    X_train = X_train.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    X_test = X_test.astype({
        'Gene_encoded': 'category',
        'TF_encoded': 'category',
        'Tissue_encoded': 'category'
    })

    rf_model = RandomForestRegressor(n_estimators=50, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error')

    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    r2_test = r2_score(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)

    test_r2_scores.append(r2_test)
    test_mse_scores.append(mse_test)

    # Save outputs from final iteration
    if i == num_iterations - 1:
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange'] = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene'] = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF'] = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene'] = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF'] = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# Summary
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²: Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²: Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE: Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# ── Save outputs ──────────────────────────────────────────────────────────────
if final_predictions_df is not None:
    final_predictions_df.to_csv("random_forest_predictions_with_tf_family.csv", index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv("random_forest_training_set.csv", index=False)
    print("✅ Saved: random_forest_training_set.csv")

# ── Save per-iteration metrics summary table ──────────────────────────────────
metrics_df = pd.DataFrame({
    'Iteration':       list(range(1, num_iterations + 1)),
    'CV_R2':           cv_r2_scores,
    'CV_MSE':          cv_mse_scores,
    'Test_R2':         test_r2_scores,
    'Test_MSE':        test_mse_scores
})

# Append a summary row
summary_row = pd.DataFrame([{
    'Iteration': 'Mean ± Std',
    'CV_R2':  f"{np.mean(cv_r2_scores):.4f} ± {np.std(cv_r2_scores):.4f}",
    'CV_MSE': f"{np.mean(cv_mse_scores):.4f} ± {np.std(cv_mse_scores):.4f}",
    'Test_R2':  f"{np.mean(test_r2_scores):.4f} ± {np.std(test_r2_scores):.4f}",
    'Test_MSE': f"{np.mean(test_mse_scores):.4f} ± {np.std(test_mse_scores):.4f}",
}])

metrics_df = pd.concat([metrics_df, summary_row], ignore_index=True)
metrics_df.to_csv("iteration_metrics_summary.csv", index=False)
print("✅ Saved: iteration_metrics_summary.csv")

# ── Boxplots: 2×2 grid (R² + MSE) ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.boxplot(y=cv_r2_scores, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title("Cross-Validation R² Distribution")
axes[0, 0].set_ylabel("R²")

sns.boxplot(y=test_r2_scores, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title("Test R² Distribution")
axes[0, 1].set_ylabel("R²")

sns.boxplot(y=cv_mse_scores, ax=axes[1, 0], color='salmon')   # ← NEW
axes[1, 0].set_title("Cross-Validation MSE Distribution")
axes[1, 0].set_ylabel("MSE")

sns.boxplot(y=test_mse_scores, ax=axes[1, 1], color='salmon')  # ← NEW
axes[1, 1].set_title("Test MSE Distribution")
axes[1, 1].set_ylabel("MSE")

plt.tight_layout()
plt.savefig("boxplots_r2_mse.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: boxplots_r2_mse.png")

In [ ]:
# ── Output directory ──────────────────────────────────────────────────────────
import os
outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

# ── Boxplots: 2×2 grid (R² + MSE) ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.boxplot(y=cv_r2_scores, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title("Cross-Validation R² Distribution")
axes[0, 0].set_ylabel("R²")

sns.boxplot(y=test_r2_scores, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title("Test R² Distribution")
axes[0, 1].set_ylabel("R²")

sns.boxplot(y=cv_mse_scores, ax=axes[1, 0], color='salmon')
axes[1, 0].set_title("Cross-Validation MSE Distribution")
axes[1, 0].set_ylabel("MSE")

sns.boxplot(y=test_mse_scores, ax=axes[1, 1], color='salmon')
axes[1, 1].set_title("Test MSE Distribution")
axes[1, 1].set_ylabel("MSE")

plt.tight_layout()

# ── Save BEFORE show() ────────────────────────────────────────────────────────
png_path = os.path.join(outdir, "boxplots_r2_mse.png")
svg_path = os.path.join(outdir, "boxplots_r2_mse.svg")

plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, format='svg', bbox_inches='tight')

print(f"✅ Saved PNG → {png_path}")
print(f"✅ Saved SVG → {svg_path}")

plt.show()  # always after savefig
plt.close()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── 0) Output directory ───────────────────────────────────────────────────────
outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

# ── 1) Load & encode ──────────────────────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

gene_le   = LabelEncoder()
tf_le     = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded']   = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded']     = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# ── 2) Run 10 iterations ──────────────────────────────────────────────────────
num_iterations          = 10
cv_r2_scores            = []
cv_mse_scores           = []
test_r2_scores          = []
test_mse_scores         = []
feat_importance_records = []

final_predictions_df = None
final_training_df    = None

for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i)

    rf_model = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv  = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf,
                               scoring='neg_mean_squared_error')
    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    test_r2_scores.append(r2_score(y_test, y_pred_test))
    test_mse_scores.append(mean_squared_error(y_test, y_pred_test))

    # Feature importance
    importances = rf_model.feature_importances_
    feat_importance_records.append({
        'Iteration':           i + 1,
        'Gene_importance':     importances[0],
        'TF_importance':       importances[1],
        'CellType_importance': importances[2],
    })

    # Save final iteration outputs
    if i == num_iterations - 1:
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange']    = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene']   = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF']     = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene']   = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF']     = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# ── 3) Print summary ──────────────────────────────────────────────────────────
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²:  Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²:              Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE:             Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# ── 4) Save CSVs ──────────────────────────────────────────────────────────────
if final_predictions_df is not None:
    final_predictions_df.to_csv(
        os.path.join(outdir, "random_forest_predictions_with_tf_family.csv"), index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv(
        os.path.join(outdir, "random_forest_training_set.csv"), index=False)
    print("✅ Saved: random_forest_training_set.csv")

metrics_df = pd.DataFrame({
    'Iteration': list(range(1, num_iterations + 1)),
    'CV_R2':     cv_r2_scores,
    'CV_MSE':    cv_mse_scores,
    'Test_R2':   test_r2_scores,
    'Test_MSE':  test_mse_scores,
})
summary_row = pd.DataFrame([{
    'Iteration': 'Mean ± Std',
    'CV_R2':   f"{np.mean(cv_r2_scores):.4f} ± {np.std(cv_r2_scores):.4f}",
    'CV_MSE':  f"{np.mean(cv_mse_scores):.4f} ± {np.std(cv_mse_scores):.4f}",
    'Test_R2': f"{np.mean(test_r2_scores):.4f} ± {np.std(test_r2_scores):.4f}",
    'Test_MSE':f"{np.mean(test_mse_scores):.4f} ± {np.std(test_mse_scores):.4f}",
}])
metrics_df = pd.concat([metrics_df, summary_row], ignore_index=True)
metrics_df.to_csv(os.path.join(outdir, "iteration_metrics_summary.csv"), index=False)
print("✅ Saved: iteration_metrics_summary.csv")

# ── 5) Boxplots: 2×2 grid — white background, dots overlaid ──────────────────
sns.set_theme(style='white')
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'savefig.facecolor': 'white',
})

panels = [
    (cv_r2_scores,    'R²',  'steelblue', "Cross-Validation R² Distribution",  "Cross-Validation Folds"),
    (test_r2_scores,  'R²',  'steelblue', "Test R² Distribution",               "Test Set"),
    (cv_mse_scores,   'MSE', 'salmon',    "Cross-Validation MSE Distribution",  "Cross-Validation Folds"),
    (test_mse_scores, 'MSE', 'salmon',    "Test MSE Distribution",               "Test Set"),
]

# 2x3 grid: top row = CV R², Test R², Feature Importance
#           bottom row = CV MSE, Test MSE, hidden
fig, axes = plt.subplots(2, 3, figsize=(18, 9), facecolor='white')

# ── Boxplot panels ────────────────────────────────────────────────────────────
box_axes = [axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]]
for ax, (scores, ylabel, color, title, xlabel) in zip(box_axes, panels):
    df_tmp = pd.DataFrame({xlabel: scores})

    sns.boxplot(
        data=df_tmp, y=xlabel, ax=ax,
        color=color, width=0.45,
        boxprops=dict(alpha=0.7),
        flierprops=dict(marker='o', markerfacecolor='none', markersize=6),
    )
    sns.stripplot(
        data=df_tmp, y=xlabel, ax=ax,
        color='black', size=7, jitter=True, alpha=0.8, zorder=3,
    )

    ax.set_facecolor('white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xticks([0])
    ax.set_xticklabels([xlabel], fontsize=10)
    sns.despine(ax=ax, trim=True)

# ── Feature importance bar chart (top right) ──────────────────────────────────
ax_fi = axes[0, 2]

# Compute mean importance across all iterations from feat_importance_records
fi_df = pd.DataFrame(feat_importance_records)
fi_means = fi_df[['Gene_importance', 'TF_importance', 'CellType_importance']].mean()
fi_stds  = fi_df[['Gene_importance', 'TF_importance', 'CellType_importance']].std()

features    = ['Gene', 'TF Motif', 'Cell Type']
colors_fi   = ['#4C72B0', '#55A868', '#C44E52']
bars = ax_fi.barh(
    features,
    fi_means.values,
    xerr=fi_stds.values,
    color=colors_fi,
    alpha=0.8,
    edgecolor='white',
    capsize=4,
    error_kw=dict(elinewidth=1.2, ecolor='black'),
)

# Annotate percentages on bars
for bar, val in zip(bars, fi_means.values):
    ax_fi.text(
        val + 0.005, bar.get_y() + bar.get_height() / 2,
        f"{val * 100:.1f}%",
        va='center', ha='left', fontsize=10, fontweight='bold'
    )

ax_fi.set_facecolor('white')
ax_fi.set_title("Feature Importance\n(Random Forest)", fontsize=12, fontweight='bold')
ax_fi.set_xlabel("Mean Importance Score", fontsize=11)
ax_fi.set_xlim(0, fi_means.max() + 0.08)
sns.despine(ax=ax_fi, trim=True)

# ── Hide unused bottom-right panel ───────────────────────────────────────────
axes[1, 2].set_visible(False)

plt.suptitle("Random Forest Model Performance across 10 Iterations",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

# Save BEFORE show()
png_path = os.path.join(outdir, "boxplots_r2_mse.png")
svg_path = os.path.join(outdir, "boxplots_r2_mse.svg")
plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
print(f"✅ Saved PNG → {png_path}")
print(f"✅ Saved SVG → {svg_path}")
plt.show()
plt.close()

In [ ]:
plt.savefig("boxplots_r2_mse.png", dpi=150, bbox_inches='tight')
plt.savefig("boxplots_r2_mse.svg", bbox_inches='tight')   # ← ADD THIS
plt.show()
print("✅ Saved: boxplots_r2_mse.png")
print("✅ Saved: boxplots_r2_mse.svg")   # ← ADD THIS

In [ ]:
# ── Boxplots: 2×2 grid (R² + MSE) with data points ──────────────────────────
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

# Pack scores into DataFrames for seaborn
df_cv_r2   = pd.DataFrame({'R²': cv_r2_scores})
df_test_r2 = pd.DataFrame({'R²': test_r2_scores})
df_cv_mse  = pd.DataFrame({'MSE': cv_mse_scores})
df_test_mse= pd.DataFrame({'MSE': test_mse_scores})

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.set_theme(style='whitegrid')

panels = [
    (axes[0, 0], cv_r2_scores,   'R²',  'steelblue', "Cross-Validation R² Distribution"),
    (axes[0, 1], test_r2_scores,  'R²',  'steelblue', "Test R² Distribution"),
    (axes[1, 0], cv_mse_scores,  'MSE', 'salmon',    "Cross-Validation MSE Distribution"),
    (axes[1, 1], test_mse_scores, 'MSE', 'salmon',    "Test MSE Distribution"),
]

for ax, scores, ylabel, color, title in panels:
    df_tmp = pd.DataFrame({ylabel: scores})

    # Boxplot
    sns.boxplot(
        data=df_tmp, y=ylabel, ax=ax,
        color=color, width=0.45,
        boxprops=dict(alpha=0.7),
        flierprops=dict(marker='o', markerfacecolor='none', markersize=6)
    )

    # Individual points overlaid
    sns.stripplot(
        data=df_tmp, y=ylabel, ax=ax,
        color='black', size=7, jitter=True, alpha=0.8, zorder=3
    )

    # Annotate median
    median_val = pd.Series(scores).median()
    ax.axhline(median_val, color='navy', linewidth=1, linestyle='--', alpha=0.5)
    ax.text(
        0.97, median_val,
        f"median = {median_val:.4f}",
        transform=ax.get_yaxis_transform(),
        ha='right', va='bottom',
        fontsize=8, color='navy'
    )

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel("")

plt.suptitle("Random Forest Model Performance across 10 Iterations", 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

# Save BEFORE show
png_path = os.path.join(outdir, "boxplots_r2_mse.png")
svg_path = os.path.join(outdir, "boxplots_r2_mse.svg")
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, format='svg', bbox_inches='tight')
print(f"✅ Saved PNG → {png_path}")
print(f"✅ Saved SVG → {svg_path}")
plt.show()
plt.close()

In [ ]:
# ── Boxplots: 2×2 grid with white background ──────────────────────────────────
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

sns.set_theme(style='white')  # no grid lines at all
plt.rcParams.update({
    'figure.facecolor': 'white',   # outer figure background
    'axes.facecolor':   'white',   # each panel background
    'savefig.facecolor':'white',   # background when saved to file
})

panels = [
    (cv_r2_scores,    'R²',  'steelblue', "Cross-Validation R² Distribution"),
    (test_r2_scores,  'R²',  'steelblue', "Test R² Distribution"),
    (cv_mse_scores,   'MSE', 'salmon',    "Cross-Validation MSE Distribution"),
    (test_mse_scores, 'MSE', 'salmon',    "Test MSE Distribution"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), facecolor='white')

for ax, (scores, ylabel, color, title) in zip(axes.flatten(), panels):
    df_tmp = pd.DataFrame({ylabel: scores})

    sns.boxplot(
        data=df_tmp, y=ylabel, ax=ax,
        color=color, width=0.45,
        boxprops=dict(alpha=0.7),
        flierprops=dict(marker='o', markerfacecolor='none', markersize=6)
    )
    sns.stripplot(
        data=df_tmp, y=ylabel, ax=ax,
        color='black', size=7, jitter=True, alpha=0.8, zorder=3
    )

    median_val = pd.Series(scores).median()
    ax.axhline(median_val, color='navy', linewidth=1, linestyle='--', alpha=0.5)
    ax.text(
        0.97, median_val,
        f"median = {median_val:.4f}",
        transform=ax.get_yaxis_transform(),
        ha='right', va='bottom', fontsize=8, color='navy'
    )

    ax.set_facecolor('white')          # belt-and-suspenders per panel
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel("")
    sns.despine(ax=ax, trim=True)      # clean axis spines only, no background

plt.suptitle("Random Forest Model Performance across 10 Iterations",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

png_path = os.path.join(outdir, "boxplots_r2_mse.png")
svg_path = os.path.join(outdir, "boxplots_r2_mse.svg")
plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
print(f"✅ Saved PNG → {png_path}")
print(f"✅ Saved SVG → {svg_path}")
plt.show()
plt.close()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── 0) Output directory ───────────────────────────────────────────────────────
outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

# ── 1) Load & encode ──────────────────────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

gene_le   = LabelEncoder()
tf_le     = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded']   = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded']     = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# ── 2) Run 10 iterations ──────────────────────────────────────────────────────
num_iterations = 10
cv_r2_scores   = []
cv_mse_scores  = []
test_r2_scores = []
test_mse_scores= []

final_predictions_df = None
final_training_df    = None

for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i)

    rf_model = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv  = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf,
                               scoring='neg_mean_squared_error')
    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    test_r2_scores.append(r2_score(y_test, y_pred_test))
    test_mse_scores.append(mean_squared_error(y_test, y_pred_test))

    # Save final iteration outputs
    if i == num_iterations - 1:
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange']    = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene']   = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF']     = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene']   = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF']     = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# ── 3) Print summary ──────────────────────────────────────────────────────────
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²:  Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²:              Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE:             Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# ── 4) Save CSVs ──────────────────────────────────────────────────────────────
if final_predictions_df is not None:
    final_predictions_df.to_csv(
        os.path.join(outdir, "random_forest_predictions_with_tf_family.csv"), index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv(
        os.path.join(outdir, "random_forest_training_set.csv"), index=False)
    print("✅ Saved: random_forest_training_set.csv")

metrics_df = pd.DataFrame({
    'Iteration': list(range(1, num_iterations + 1)),
    'CV_R2':     cv_r2_scores,
    'CV_MSE':    cv_mse_scores,
    'Test_R2':   test_r2_scores,
    'Test_MSE':  test_mse_scores,
})
summary_row = pd.DataFrame([{
    'Iteration': 'Mean ± Std',
    'CV_R2':   f"{np.mean(cv_r2_scores):.4f} ± {np.std(cv_r2_scores):.4f}",
    'CV_MSE':  f"{np.mean(cv_mse_scores):.4f} ± {np.std(cv_mse_scores):.4f}",
    'Test_R2': f"{np.mean(test_r2_scores):.4f} ± {np.std(test_r2_scores):.4f}",
    'Test_MSE':f"{np.mean(test_mse_scores):.4f} ± {np.std(test_mse_scores):.4f}",
}])
metrics_df = pd.concat([metrics_df, summary_row], ignore_index=True)
metrics_df.to_csv(os.path.join(outdir, "iteration_metrics_summary.csv"), index=False)
print("✅ Saved: iteration_metrics_summary.csv")

# ── 5) Boxplots: 2×2 grid — white background, dots overlaid ──────────────────
sns.set_theme(style='white')
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'savefig.facecolor': 'white',
})

panels = [
    (cv_r2_scores,    'R²',  'steelblue', "Cross-Validation R² Distribution"),
    (test_r2_scores,  'R²',  'steelblue', "Test R² Distribution"),
    (cv_mse_scores,   'MSE', 'salmon',    "Cross-Validation MSE Distribution"),
    (test_mse_scores, 'MSE', 'salmon',    "Test MSE Distribution"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), facecolor='white')

for ax, (scores, ylabel, color, title) in zip(axes.flatten(), panels):
    df_tmp = pd.DataFrame({ylabel: scores})

    sns.boxplot(
        data=df_tmp, y=ylabel, ax=ax,
        color=color, width=0.45,
        boxprops=dict(alpha=0.7),
        flierprops=dict(marker='o', markerfacecolor='none', markersize=6),
    )
    sns.stripplot(
        data=df_tmp, y=ylabel, ax=ax,
        color='black', size=7, jitter=True, alpha=0.8, zorder=3,
    )

    ax.set_facecolor('white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xlabel("")
    sns.despine(ax=ax, trim=True)

plt.suptitle("Random Forest Model Performance across 10 Iterations",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

# Save BEFORE show()
png_path = os.path.join(outdir, "boxplots_r2_mse.png")
svg_path = os.path.join(outdir, "boxplots_r2_mse.svg")
plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
print(f"✅ Saved PNG → {png_path}")
print(f"✅ Saved SVG → {svg_path}")
plt.show()
plt.close()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── 0) Output directory ───────────────────────────────────────────────────────
outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

# ── 1) Load & encode ──────────────────────────────────────────────────────────
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

gene_le   = LabelEncoder()
tf_le     = LabelEncoder()
tissue_le = LabelEncoder()

df_model['Gene_encoded']   = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded']     = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# ── 2) Run 10 iterations ──────────────────────────────────────────────────────
num_iterations = 10
cv_r2_scores   = []
cv_mse_scores  = []
test_r2_scores = []
test_mse_scores= []

final_predictions_df = None
final_training_df    = None

for i in range(num_iterations):
    print(f"\n🔁 Iteration {i + 1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)

    X = df_sample[['Gene_encoded', 'TF_encoded', 'Tissue_encoded']]
    y = df_sample['log2FoldChange']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=i)

    rf_model = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=i)
    kf = KFold(n_splits=5, shuffle=True, random_state=i)

    # Cross-validation
    r2_cv  = cross_val_score(rf_model, X_train, y_train, cv=kf, scoring='r2')
    mse_cv = -cross_val_score(rf_model, X_train, y_train, cv=kf,
                               scoring='neg_mean_squared_error')
    cv_r2_scores.append(np.mean(r2_cv))
    cv_mse_scores.append(np.mean(mse_cv))

    # Train + test
    rf_model.fit(X_train, y_train)
    y_pred_test = rf_model.predict(X_test)

    test_r2_scores.append(r2_score(y_test, y_pred_test))
    test_mse_scores.append(mean_squared_error(y_test, y_pred_test))

    # Save final iteration outputs
    if i == num_iterations - 1:
        tf_family_map = df_model[['TF', 'TF_Family']].drop_duplicates()

        X_test_copy = X_test.copy()
        X_test_copy['Actual_log2FoldChange']    = y_test.values
        X_test_copy['Predicted_log2FoldChange'] = y_pred_test
        X_test_copy['Gene']   = gene_le.inverse_transform(X_test_copy['Gene_encoded'])
        X_test_copy['TF']     = tf_le.inverse_transform(X_test_copy['TF_encoded'])
        X_test_copy['Tissue'] = tissue_le.inverse_transform(X_test_copy['Tissue_encoded'])
        X_test_copy = X_test_copy.merge(tf_family_map, on='TF', how='left')
        final_predictions_df = X_test_copy

        X_train_copy = X_train.copy()
        X_train_copy['log2FoldChange'] = y_train.values
        X_train_copy['Gene']   = gene_le.inverse_transform(X_train_copy['Gene_encoded'])
        X_train_copy['TF']     = tf_le.inverse_transform(X_train_copy['TF_encoded'])
        X_train_copy['Tissue'] = tissue_le.inverse_transform(X_train_copy['Tissue_encoded'])
        X_train_copy = X_train_copy.merge(tf_family_map, on='TF', how='left')
        final_training_df = X_train_copy

# ── 3) Print summary ──────────────────────────────────────────────────────────
print("\n📈 Summary over all iterations:")
print(f"Cross-Validation R²:  Mean = {np.mean(cv_r2_scores):.4f}, Std = {np.std(cv_r2_scores):.4f}")
print(f"Cross-Validation MSE: Mean = {np.mean(cv_mse_scores):.4f}, Std = {np.std(cv_mse_scores):.4f}")
print(f"Test R²:              Mean = {np.mean(test_r2_scores):.4f}, Std = {np.std(test_r2_scores):.4f}")
print(f"Test MSE:             Mean = {np.mean(test_mse_scores):.4f}, Std = {np.std(test_mse_scores):.4f}")

# ── 4) Save CSVs ──────────────────────────────────────────────────────────────
if final_predictions_df is not None:
    final_predictions_df.to_csv(
        os.path.join(outdir, "random_forest_predictions_with_tf_family.csv"), index=False)
    print("✅ Saved: random_forest_predictions_with_tf_family.csv")

if final_training_df is not None:
    final_training_df.to_csv(
        os.path.join(outdir, "random_forest_training_set.csv"), index=False)
    print("✅ Saved: random_forest_training_set.csv")

metrics_df = pd.DataFrame({
    'Iteration': list(range(1, num_iterations + 1)),
    'CV_R2':     cv_r2_scores,
    'CV_MSE':    cv_mse_scores,
    'Test_R2':   test_r2_scores,
    'Test_MSE':  test_mse_scores,
})
summary_row = pd.DataFrame([{
    'Iteration': 'Mean ± Std',
    'CV_R2':   f"{np.mean(cv_r2_scores):.4f} ± {np.std(cv_r2_scores):.4f}",
    'CV_MSE':  f"{np.mean(cv_mse_scores):.4f} ± {np.std(cv_mse_scores):.4f}",
    'Test_R2': f"{np.mean(test_r2_scores):.4f} ± {np.std(test_r2_scores):.4f}",
    'Test_MSE':f"{np.mean(test_mse_scores):.4f} ± {np.std(test_mse_scores):.4f}",
}])
metrics_df = pd.concat([metrics_df, summary_row], ignore_index=True)
metrics_df.to_csv(os.path.join(outdir, "iteration_metrics_summary.csv"), index=False)
print("✅ Saved: iteration_metrics_summary.csv")

# ── 5) Boxplots: 2×2 grid — white background, dots overlaid ──────────────────
sns.set_theme(style='white')
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'savefig.facecolor': 'white',
})

panels = [
    (cv_r2_scores,    'R²',  'steelblue', "Cross-Validation R² Distribution",  "Cross-Validation Folds"),
    (test_r2_scores,  'R²',  'steelblue', "Test R² Distribution",               "Test Set"),
    (cv_mse_scores,   'MSE', 'salmon',    "Cross-Validation MSE Distribution",  "Cross-Validation Folds"),
    (test_mse_scores, 'MSE', 'salmon',    "Test MSE Distribution",               "Test Set"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9), facecolor='white')

for ax, (scores, ylabel, color, title, xlabel) in zip(axes.flatten(), panels):
    df_tmp = pd.DataFrame({xlabel: scores})

    sns.boxplot(
        data=df_tmp, y=xlabel, ax=ax,
        color=color, width=0.45,
        boxprops=dict(alpha=0.7),
        flierprops=dict(marker='o', markerfacecolor='none', markersize=6),
    )
    sns.stripplot(
        data=df_tmp, y=xlabel, ax=ax,
        color='black', size=7, jitter=True, alpha=0.8, zorder=3,
    )

    ax.set_facecolor('white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xticks([0])
    ax.set_xticklabels([xlabel], fontsize=10)
    sns.despine(ax=ax, trim=True)

plt.suptitle("Random Forest Model Performance across 10 Iterations",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()

# Save BEFORE show()
png_path = os.path.join(outdir, "boxplots_r2_mse.png")
svg_path = os.path.join(outdir, "boxplots_r2_mse.svg")
plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
print(f"✅ Saved PNG → {png_path}")
print(f"✅ Saved SVG → {svg_path}")
plt.show()
plt.close()

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
outdir = "machinelearning_TF_genes/analysis_outputs"
os.makedirs(outdir, exist_ok=True)

# ------------------------------------------------------------
# 1) Load & encode
# ------------------------------------------------------------
df = pd.read_csv("final_filtered_gene_tf_data.csv")
df_model = df[['Gene', 'TF', 'Tissue', 'log2FoldChange', 'TF_Family']].dropna()

gene_le    = LabelEncoder()
tf_le      = LabelEncoder()
tissue_le  = LabelEncoder()

df_model['Gene_encoded']   = gene_le.fit_transform(df_model['Gene'])
df_model['TF_encoded']     = tf_le.fit_transform(df_model['TF'])
df_model['Tissue_encoded'] = tissue_le.fit_transform(df_model['Tissue'])

# Composite gene-motif interaction feature
df_model['Gene_TF_encoded'] = (
    df_model['Gene_encoded'].astype(str) + "_" + df_model['TF_encoded'].astype(str)
)
composite_le = LabelEncoder()
df_model['Gene_TF_encoded'] = composite_le.fit_transform(df_model['Gene_TF_encoded'])

# ------------------------------------------------------------
# 2) Define the 5 feature configurations
# ------------------------------------------------------------
feature_sets = {
    'Gene only':           ['Gene_encoded'],
    'Gene + TF':           ['Gene_encoded', 'TF_encoded'],
    'Composite only':      ['Gene_TF_encoded'],
    'Full':                ['Gene_encoded', 'TF_encoded', 'Tissue_encoded'],
    'Composite + Tissue':  ['Gene_TF_encoded', 'Tissue_encoded'],
}

# ------------------------------------------------------------
# 3) Run 10 iterations per feature set
# ------------------------------------------------------------
num_iterations = 10
results = {name: [] for name in feature_sets}

for i in range(num_iterations):
    print(f"Iteration {i+1}/{num_iterations}")
    df_sample = df_model.sample(n=20000, random_state=i)
    y = df_sample['log2FoldChange']

    for name, features in feature_sets.items():
        X = df_sample[features]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=i
        )
        rf = RandomForestRegressor(n_estimators=500, random_state=i, n_jobs=-1)
        rf.fit(X_train, y_train)
        r2 = r2_score(y_test, rf.predict(X_test))
        results[name].append(r2)
        print(f"  {name}: R²={r2:.4f}")

# ------------------------------------------------------------
# 4) Build df_plot ordered by median R²
# ------------------------------------------------------------
df_plot = pd.DataFrame(results)
order_plot = df_plot.median().sort_values().index.tolist()

# ------------------------------------------------------------
# 5) Plot
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.set_theme(style='whitegrid')
sns.boxplot(data=df_plot, order=order_plot, palette='pastel')
sns.stripplot(data=df_plot, order=order_plot, color='gray', size=3, jitter=True)

# Annotate medians inside boxes
for i, col in enumerate(order_plot):
    median_val = float(df_plot[col].median())
    plt.text(
        i, median_val, f"{median_val:.3f}",
        ha='center', va='center',
        fontsize=9, fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.7, boxstyle='round,pad=0.2')
    )

plt.ylabel("Test $R^2$", fontsize=12)
plt.title("Feature-set Comparison: Test $R^2$ Distribution", fontsize=14)
plt.xticks(rotation=45, fontsize=10)
plt.ylim(df_plot.min().min() - 0.05, df_plot.max().max() + 0.05)
sns.despine(trim=True)
plt.tight_layout()

# Save figure
svg_path = os.path.join(outdir, "feature_set_comparison_r2.svg")
png_path = os.path.join(outdir, "feature_set_comparison_r2.png")
plt.savefig(svg_path, format="svg", bbox_inches="tight")
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved SVG → {svg_path}")
print(f"Saved PNG → {png_path}")

# ------------------------------------------------------------
# 6) Save supplementary tables
# ------------------------------------------------------------
# Raw per-iteration R² values
raw_path = os.path.join(outdir, "supplementary_figure2c_raw_r2.csv")
df_plot.to_csv(raw_path, index_label="iteration")
print(f"Saved raw R² table → {raw_path}")

# Summary stats
summary = df_plot.agg(['mean', 'std', 'min', 'max']).T
summary.index.name = "feature_set"
summary.columns = ["R2_mean", "R2_std", "R2_min", "R2_max"]
summary_path = os.path.join(outdir, "supplementary_figure2c_summary.csv")
summary.to_csv(summary_path)
print(f"Saved summary → {summary_path}")

# Excel with both sheets
excel_path = os.path.join(outdir, "supplementary_figure2c_data.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_plot.to_excel(writer, sheet_name="Raw_R2_per_iteration", index_label="iteration")
    summary.to_excel(writer, sheet_name="Summary_stats")
print(f"Saved Excel → {excel_path}")